# Voluntary Cancellations: analysis and PowerPoint in one notebook

**Part 1 (R: sparklyr / dplyr / ggplot2)** does every calculation on the start-status basis (each subscription is New or Win-back by the status it started with) and saves small summary tables as CSV.
**Part 2 (Python: matplotlib + python-pptx)** reads those tables, draws the charts in grey / light grey, and builds the TOD-format deck. Slide titles and cards are written from the numbers, so they update on every run.

**How to run**
1. Set the widgets at the top (window `FROM` / `TO`, `ASOF`, table names). Defaults reproduce the Aug-Sep 2026 analysis.
2. *Run all*. The last cell shows a download link to the `.pptx`.
3. `CHART_SOURCE`: `python` (default) draws the deck charts with matplotlib; `ggplot2` puts the R charts from cell R 9 into the deck instead.

| Widget | Meaning |
|---|---|
| `FROM`, `TO` | Cancellation window under review |
| `HIST_FROM` | Start of history for the run rate and monthly charts |
| `EVENT_FROM` | Start of the match-day analysis (peak days, competitions, matches, World Cup) |
| `ASOF` | Latest date in the data; `MATURE_DAYS` = days after expiry before a return is read |
| `VIEW_TBL`, `SUB_TBL`, `TXN_TBL` | Viewing, subscriber and transaction tables / views (`*_PATH` = parquet used only if the view is missing) |
| `SUFFIX_LEN` | Viewing `subscriptionid` = Subscription_ID + this many digits |
| `OUT_ROOT` | Folder for tables (`/data`), charts (`/charts`) and the PPT. On Unity Catalog clusters that block `/dbfs/FileStore`, use a Volume, e.g. `/Volumes/<catalog>/<schema>/<volume>/cancellation_ppt` |

Slides whose table is not available (no transaction table, no World Cup in the window) are skipped automatically. Football-driven cycling and product or commercial problems remain hypotheses, not proven causes.

## Part 1: calculations in R (sparklyr / dplyr) and ggplot2 charts

In [0]:
library(sparklyr); library(dplyr); library(tidyr); library(ggplot2); library(scales)
# SparkR is attached above dplyr in Databricks and masks these verbs; point them back to dplyr
for (f in c("collect", "filter", "mutate", "arrange", "distinct", "group_by", "select", "desc", "row_number", "coalesce",
            "summarise", "summarize", "rename", "n", "count", "between", "lag", "lead"))
  if (f %in% getNamespaceExports("dplyr")) assign(f, getExportedValue("dplyr", f), envir = globalenv())

sc <- spark_connect(method = "databricks")

w <- function(name, default, label = name) { dbutils.widgets.text(name, default, label); dbutils.widgets.get(name) }
dbutils.widgets.dropdown("CHART_SOURCE", "python", c("python", "ggplot2"), "Charts in the PPT drawn by")
P <- list(
  FROM        = w("FROM", "2026-08-01", "Cancellation window from"),
  TO          = w("TO", "2026-09-30", "Cancellation window to"),
  HIST_FROM   = w("HIST_FROM", "2025-01-01", "History from (run rate)"),
  EVENT_FROM  = w("EVENT_FROM", "2026-01-01", "Match-day analysis from"),
  ASOF        = w("ASOF", "2026-10-13", "Latest date in the data"),
  MATURE_DAYS = w("MATURE_DAYS", "30", "Days after expiry before 'returned?' is read"),
  VIEW_TBL    = w("VIEW_TBL", "EntertainmentMovies_SD", "Viewing table / view"),
  SUFFIX_LEN  = w("SUFFIX_LEN", "6", "Viewing subscriptionid suffix length"),
  SUB_TBL     = w("SUB_TBL", "DIM_SubscribersInfo_SDF", "Subscriber table / view"),
  TXN_TBL     = w("TXN_TBL", "DIM_Transactions_SD", "Transaction table / view"),
  VIEW_PATH   = w("VIEW_PATH", "dbfs:/mnt/viewershipinfo/parquet/ALL_V2/EntertainmentMovies_SD", "Viewing parquet (if view missing)"),
  SUB_PATH    = w("SUB_PATH", "dbfs:/mnt/dimsubscriber/parquet/ALL/DIMSubscriber", "Subscriber parquet (if view missing)"),
  TXN_PATH    = w("TXN_PATH", "", "Transaction parquet (if view missing)"),
  OUT_ROOT    = w("OUT_ROOT", "/dbfs/FileStore/cancellation_ppt", "Output folder (tables, charts, PPT)"),
  CHART_SOURCE = dbutils.widgets.get("CHART_SOURCE")
)
DATA_DIR  <- file.path(P$OUT_ROOT, "data")
CHART_DIR <- file.path(P$OUT_ROOT, "charts")
dir.create(DATA_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(CHART_DIR, recursive = TRUE, showWarnings = FALSE)
unlink(list.files(DATA_DIR, pattern = "\\.csv$", full.names = TRUE))   # no stale tables from an earlier run

fill <- function(s, p = P) { for (nm in names(p)) s <- gsub(paste0("{", nm, "}"), p[[nm]], s, fixed = TRUE); s }
q    <- function(sql) sdf_sql(sc, fill(sql))
reg  <- function(sdf, name) { sparklyr::sdf_register(sdf, name); sdf }

# Every summary table the PPT needs is saved as CSV; the Python cells read them back
save_tbl <- function(x, name) {
  if (inherits(x, "tbl_spark")) x <- dplyr::collect(x)
  x <- as.data.frame(x)
  utils::write.csv(x, file.path(DATA_DIR, paste0(name, ".csv")), row.names = FALSE, na = "")
  cat(sprintf("saved %-20s %7d rows\n", name, nrow(x)))
  invisible(x)
}
save_kv <- function(x, name) save_tbl(data.frame(key = names(x), value = vapply(x, function(v) as.character(v)[1], ""), stringsAsFactors = FALSE), name)

view_exists <- function(name) tryCatch({ invisible(sdf_sql(sc, paste0("SELECT * FROM ", name, " LIMIT 0"))); TRUE }, error = function(e) FALSE)
ensure_view <- function(name, path) {
  if (view_exists(name)) return(TRUE)
  if (is.null(path) || path == "") return(FALSE)
  invisible(spark_read_parquet(sc, name = name, path = path, memory = FALSE))
  cat("registered", name, "from", path, "\n"); TRUE
}
stopifnot(ensure_view(P$SUB_TBL, P$SUB_PATH), ensure_view(P$VIEW_TBL, P$VIEW_PATH))
HAS_TXN <- ensure_view(P$TXN_TBL, P$TXN_PATH)
if (!HAS_TXN) cat("No transaction table: the revenue slide and 'charged after cancel' will be skipped\n")
save_kv(c(P, HAS_TXN = HAS_TXN, RUN_AT = format(Sys.time(), "%Y-%m-%d %H:%M")), "params")

In [0]:
base_rows <- q("
  SELECT Customer_ID, Customer_External_ID, Subscription_ID, Subscription_version, Subscription_latest,
         Subscription_type, Subscription_status, to_date(Subscription_start_date) AS start_date,
         to_date(Expiry_date) AS expiry_date, to_date(Cancellation_date) AS cancel_date,
         Cancellation_reason, churn_reason, Subscription_churn_type, Acquisition_channel, Product_name,
         Offer_period, Country, Source_system
  FROM {SUB_TBL}
  WHERE D2C_B2B = 'D2C' AND offer_type = 'Subscription' AND product_type IS NULL
    AND non_commercial <> 1 AND daily_weekly_flag <> 1
    AND Customer_External_ID IS NOT NULL AND TRIM(Customer_External_ID) <> '' AND TRIM(Customer_External_ID) <> 'null'") %>%
  reg("base_rows")

first_v <- q("
  SELECT Subscription_ID, Subscription_type AS start_status FROM (
    SELECT Subscription_ID, Subscription_type,
           ROW_NUMBER() OVER (PARTITION BY Subscription_ID ORDER BY Subscription_version ASC) rn
    FROM base_rows WHERE Subscription_type IN ('new','winback')) WHERE rn = 1") %>% reg("first_v")

life <- q("
  WITH l AS (SELECT * FROM base_rows WHERE Subscription_latest = 1),
  o AS (
    SELECT l.*,
      ROW_NUMBER() OVER (PARTITION BY Customer_External_ID ORDER BY start_date, Subscription_ID) AS seq,
      MAX(expiry_date) OVER (PARTITION BY Customer_External_ID ORDER BY start_date, Subscription_ID
                             ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS prev_paid_through,
      LEAD(start_date) OVER (PARTITION BY Customer_External_ID ORDER BY start_date, Subscription_ID) AS next_start,
      LEAD(Acquisition_channel) OVER (PARTITION BY Customer_External_ID ORDER BY start_date, Subscription_ID) AS next_channel,
      LEAD(Subscription_ID) OVER (PARTITION BY Customer_External_ID ORDER BY start_date, Subscription_ID) AS next_sub_id
    FROM l)
  SELECT o.*, COALESCE(f.start_status, CASE WHEN seq = 1 THEN 'new' ELSE 'winback' END) AS start_status,
         CASE WHEN seq = 1 THEN 'new' ELSE 'winback' END AS rule_status,
         CASE WHEN Subscription_status = 'churned' THEN 'churned' ELSE 'active' END AS end_status,
         datediff(expiry_date, start_date)  AS paid_period_days,
         datediff(cancel_date, start_date)  AS days_start_to_cancel,
         datediff(expiry_date, cancel_date) AS days_cancel_to_expiry,
         datediff(start_date, prev_paid_through) AS gap_since_prev_paid_through,
         datediff(next_start, expiry_date)  AS days_expiry_to_next_start,
         CASE WHEN cancel_date IS NULL THEN 'No cancel date'
              WHEN datediff(expiry_date, cancel_date) = 0 THEN 'Cancelled on expiry day'
              WHEN datediff(cancel_date, start_date) <= 1 THEN 'Cancelled on purchase day / next day'
              ELSE 'Cancelled mid-period' END AS cancel_type
  FROM o LEFT JOIN first_v f USING (Subscription_ID)") %>% reg("life")
tbl_cache(sc, "life")

display(q("SELECT start_status = rule_status AS agree, COUNT(*) n FROM life GROUP BY 1"))
display(q("SELECT start_status, end_status, COUNT(*) subs FROM life GROUP BY 1,2 ORDER BY 1,2"))

cohort <- q("SELECT * FROM life WHERE cancel_date >= '{FROM}' AND cancel_date <= '{TO}'") %>% reg("cohort")
tbl_cache(sc, "cohort")
cat("Cohort subscriptions:", sdf_nrow(cohort), "\n")

In [0]:
kpi     <- q("SELECT start_status, COUNT(*) AS subs, COUNT(DISTINCT Customer_External_ID) AS customers FROM cohort GROUP BY start_status") %>% collect()
kpi_all <- q("SELECT 'all' AS start_status, COUNT(*) AS subs, COUNT(DISTINCT Customer_External_ID) AS customers FROM cohort") %>% collect()
save_tbl(bind_rows(kpi, kpi_all), "kpi")

monthly <- q("
  SELECT date_format(cancel_date,'yyyy-MM') AS m, start_status, COUNT(*) AS subs, COUNT(DISTINCT Customer_External_ID) AS customers
  FROM life WHERE cancel_date >= '{HIST_FROM}' AND cancel_date <= '{TO}' GROUP BY 1,2") %>% collect() %>% arrange(m, start_status)
save_tbl(monthly, "monthly")

tot  <- monthly %>% group_by(m) %>% summarise(customers = sum(customers), .groups = "drop")
base <- tot %>% filter(m < substr(P$FROM, 1, 7)); last_m <- substr(P$TO, 1, 7)
if (nrow(base) > 1 && last_m %in% tot$m)
  cat(sprintf("Run rate %s to %s: mean %.0f, sd %.0f. %s: %d (z = %.2f)\n", min(base$m), max(base$m), mean(base$customers), sd(base$customers),
              last_m, as.integer(tot$customers[tot$m == last_m]), (tot$customers[tot$m == last_m] - mean(base$customers)) / sd(base$customers)))

display(q("SELECT start_status, cancel_type, COUNT(*) subs, ROUND(AVG(paid_period_days),1) avg_paid_period FROM cohort GROUP BY 1,2 ORDER BY 1,2"))

if (HAS_TXN) {
  last_charge <- q("
    SELECT Subscription_ID, Transaction_price_incl_TAX_USD AS last_charge_usd, Payment_type FROM (
      SELECT Subscription_ID, Transaction_price_incl_TAX_USD, Payment_type,
             ROW_NUMBER() OVER (PARTITION BY Subscription_ID ORDER BY Transaction_Date DESC) rn FROM {TXN_TBL}) WHERE rn = 1") %>%
    reg("last_charge")
  rev <- q("SELECT date_format(c.cancel_date,'yyyy-MM') AS m, c.start_status, COUNT(*) AS subs,
                   ROUND(SUM(l.last_charge_usd),0) AS renewal_value_forgone_usd, ROUND(AVG(l.last_charge_usd),2) AS avg_last_charge
            FROM life c LEFT JOIN last_charge l USING (Subscription_ID)
            WHERE c.cancel_date >= '{HIST_FROM}' AND c.cancel_date <= '{TO}' GROUP BY 1,2") %>% collect() %>% arrange(m, start_status)
  save_tbl(rev, "rev")
}

cancel_speed <- q("
  SELECT CASE WHEN cancel_date >= '{FROM}' THEN 'Review window' ELSE 'History' END AS period, start_status, COUNT(*) AS subs,
         ROUND(AVG(CASE WHEN days_start_to_cancel <= 1  THEN 1.0 ELSE 0 END),3) AS within_1d,
         ROUND(AVG(CASE WHEN days_start_to_cancel <= 7  THEN 1.0 ELSE 0 END),3) AS within_7d,
         ROUND(AVG(CASE WHEN days_start_to_cancel <= 30 THEN 1.0 ELSE 0 END),3) AS within_30d,
         percentile_approx(days_start_to_cancel, 0.5) AS median_days
  FROM life WHERE cancel_date >= '{HIST_FROM}' AND cancel_date <= '{TO}' GROUP BY 1,2")
save_tbl(cancel_speed, "cancel_speed")

In [0]:
v <- q("
  SELECT viewerId, to_date(startdate) AS vdate, convivasessionid, subscriptionid,
         CAST(substr(CAST(subscriptionid AS STRING), 1, length(CAST(subscriptionid AS STRING)) - {SUFFIX_LEN}) AS BIGINT) AS sub_id_view,
         playingunixtimems / 60000.0 AS minutes, startuperror, errorlist, interrupts, bufferingunixtimems / 60000.0 AS buffer_min,
         competitionName, ContentType2, Live, to_date(match_date) AS match_date, deviceos, planname, asset_id,
         CASE WHEN errorlist IS NOT NULL OR startuperror = 1 THEN 1 ELSE 0 END AS has_error,
         CASE WHEN errorlist LIKE '%ntitlement%' THEN 1 ELSE 0 END AS entitlement_error
  FROM {VIEW_TBL}
  WHERE viewerId IN (SELECT DISTINCT Customer_External_ID FROM cohort)") %>% reg("v")
tbl_cache(sc, "v")

sub_view <- q("
  SELECT c.Subscription_ID, c.Customer_External_ID, c.start_status, c.cancel_type, c.start_date, c.cancel_date, c.expiry_date,
         c.Product_name, c.Country, c.Acquisition_channel, c.Cancellation_reason, c.churn_reason, c.days_start_to_cancel,
         COUNT(DISTINCT CASE WHEN v.sub_id_view = c.Subscription_ID THEN v.convivasessionid END) AS own_sessions,
         COUNT(DISTINCT CASE WHEN v.sub_id_view = c.Subscription_ID AND v.minutes >= 1 THEN v.convivasessionid END) AS own_played_sessions,
         COALESCE(SUM(CASE WHEN v.sub_id_view = c.Subscription_ID THEN v.minutes END),0) AS own_minutes,
         COUNT(DISTINCT CASE WHEN v.sub_id_view = c.Subscription_ID THEN v.vdate END) AS own_days,
         COUNT(DISTINCT CASE WHEN v.sub_id_view = c.Subscription_ID THEN v.asset_id END) AS own_assets,
         SUM(CASE WHEN v.sub_id_view = c.Subscription_ID THEN v.has_error ELSE 0 END) AS own_error_sessions,
         SUM(CASE WHEN v.sub_id_view = c.Subscription_ID THEN v.entitlement_error ELSE 0 END) AS own_entitlement_errors,
         COUNT(DISTINCT CASE WHEN v.vdate BETWEEN c.start_date AND COALESCE(c.cancel_date, c.expiry_date) THEN v.convivasessionid END) AS any_sessions_in_window,
         COALESCE(SUM(CASE WHEN v.vdate BETWEEN c.start_date AND COALESCE(c.cancel_date, c.expiry_date) THEN v.minutes END),0) AS any_minutes_in_window,
         COUNT(DISTINCT CASE WHEN v.vdate BETWEEN c.start_date AND COALESCE(c.cancel_date, c.expiry_date) THEN v.asset_id END) AS any_assets_in_window,
         COUNT(DISTINCT CASE WHEN v.vdate > c.cancel_date AND v.vdate <= c.expiry_date THEN v.convivasessionid END) AS sessions_cancel_to_expiry,
         COALESCE(SUM(CASE WHEN v.vdate > c.cancel_date AND v.vdate <= c.expiry_date THEN v.minutes END),0) AS minutes_cancel_to_expiry,
         COUNT(DISTINCT CASE WHEN v.vdate > date_sub(c.expiry_date,7) AND v.vdate <= c.expiry_date THEN v.vdate END) AS days_in_final_week,
         COALESCE(SUM(CASE WHEN v.vdate > c.expiry_date AND (c.next_start IS NULL OR v.vdate < c.next_start) THEN v.minutes END),0) AS minutes_after_expiry
  FROM cohort c LEFT JOIN v ON c.Customer_External_ID = v.viewerId
  GROUP BY c.Subscription_ID, c.Customer_External_ID, c.start_status, c.cancel_type, c.start_date, c.cancel_date, c.expiry_date,
           c.Product_name, c.Country, c.Acquisition_channel, c.Cancellation_reason, c.churn_reason, c.days_start_to_cancel, c.next_start") %>%
  reg("sub_view")

tiers <- q("
  SELECT *, CASE WHEN any_sessions_in_window = 0 THEN '0 no session in window'
                 WHEN any_minutes_in_window < 1 THEN '1 tried, under 1 minute'
                 WHEN any_minutes_in_window < 60 THEN '2 1-59 min'
                 WHEN any_minutes_in_window < 300 THEN '3 1-5 hours'
                 WHEN any_minutes_in_window < 1200 THEN '4 5-20 hours' ELSE '5 20+ hours' END AS mou_tier,
             CASE WHEN own_sessions = 0 AND any_sessions_in_window > 0 THEN 'watched only on another subscription / pass'
                  WHEN own_minutes >= 1 THEN 'watched this subscription' ELSE 'did not watch this subscription' END AS attribution
  FROM sub_view") %>% reg("tiers")
tbl_cache(sc, "tiers")

save_tbl(q("SELECT start_status, mou_tier, attribution, COUNT(*) AS subs, ROUND(AVG(any_minutes_in_window),0) AS avg_min
            FROM tiers GROUP BY 1,2,3 ORDER BY 1,2,3"), "tiers")
save_tbl(q("SELECT start_status, COUNT(*) AS subs, SUM(CASE WHEN own_minutes < 1 AND any_minutes_in_window < 1 THEN 1 ELSE 0 END) AS never
            FROM tiers GROUP BY 1"), "never_watched")

nw <- q("SELECT * FROM tiers WHERE own_minutes < 1 AND any_minutes_in_window < 1") %>% reg("nw")
save_tbl(q("
  SELECT n.start_status,
         COUNT(DISTINCT n.Subscription_ID) AS non_watching_subs,
         COUNT(DISTINCT CASE WHEN v.convivasessionid IS NULL THEN n.Subscription_ID END) AS never_opened_player,
         COUNT(DISTINCT CASE WHEN v.has_error = 1 THEN n.Subscription_ID END) AS had_error_coded_session,
         COUNT(DISTINCT CASE WHEN v.entitlement_error = 1 THEN n.Subscription_ID END) AS had_entitlement_error,
         COUNT(DISTINCT CASE WHEN v.convivasessionid IS NOT NULL AND v.has_error = 0 AND v.minutes < 1 THEN n.Subscription_ID END) AS opened_no_error_no_play
  FROM nw n LEFT JOIN v ON v.viewerId = n.Customer_External_ID AND v.vdate BETWEEN n.start_date AND COALESCE(n.cancel_date, n.expiry_date)
  GROUP BY 1"), "nw_problems")
save_tbl(q("SELECT start_status, Product_name, Country, COUNT(*) AS subs FROM nw GROUP BY 1,2,3"), "nw_product_country")
display(q("SELECT regexp_extract(errorlist, '^(.{0,40})', 1) AS error_head, deviceos, COUNT(*) sessions FROM v WHERE has_error = 1 GROUP BY 1,2 ORDER BY sessions DESC LIMIT 50"))

In [0]:
fixtures <- q("
  SELECT to_date(match_date) AS match_date, competitionName, COUNT(DISTINCT viewerId) AS viewers FROM {VIEW_TBL}
  WHERE ContentType2 = 'Football' AND to_date(match_date) >= '{HIST_FROM}' GROUP BY 1,2") %>% reg("fixtures")
tbl_cache(sc, "fixtures")
peak_md <- q("SELECT match_date FROM (SELECT match_date, SUM(viewers) v, percent_rank() OVER (ORDER BY SUM(viewers)) pr
              FROM fixtures GROUP BY 1) WHERE pr >= 0.75") %>% reg("peak_md")

ev_study <- q("
  SELECT c.start_status, datediff(c.d, f.match_date) AS offset_days,
         SUM(c.n) / (SELECT COUNT(*) FROM peak_md WHERE match_date >= '{EVENT_FROM}' AND match_date <= date_sub(DATE'{TO}', 7)) AS avg_cancels_per_peak_day
  FROM (SELECT cancel_date AS d, start_status, COUNT(*) n FROM life WHERE cancel_date >= '{EVENT_FROM}' AND cancel_date <= '{TO}' GROUP BY 1,2) c
  JOIN (SELECT match_date FROM peak_md WHERE match_date >= '{EVENT_FROM}' AND match_date <= date_sub(DATE'{TO}', 7)) f
    ON datediff(c.d, f.match_date) BETWEEN -3 AND 7
  GROUP BY 1,2") %>% collect() %>% arrange(start_status, offset_days)
save_tbl(ev_study, "ev_study")

n_peak  <- q("SELECT COUNT(*) AS n FROM peak_md WHERE match_date >= '{EVENT_FROM}' AND match_date <= date_sub(DATE'{TO}', 7)") %>% collect() %>% pull(n)
n_md    <- q("SELECT COUNT(DISTINCT match_date) AS n FROM fixtures WHERE match_date >= '{EVENT_FROM}' AND match_date <= date_sub(DATE'{TO}', 7)") %>% collect() %>% pull(n)
avg_day <- q("SELECT start_status, COUNT(*) / (datediff(DATE'{TO}', DATE'{EVENT_FROM}') + 1) AS avg_daily
              FROM life WHERE cancel_date >= '{EVENT_FROM}' AND cancel_date <= '{TO}' GROUP BY 1") %>% collect()
save_kv(list(peak_days = n_peak, match_days = n_md,
             avg_daily_new = avg_day$avg_daily[avg_day$start_status == "new"],
             avg_daily_winback = avg_day$avg_daily[avg_day$start_status == "winback"]), "event_meta")

# Daily cancellations used by the competition / match / World Cup event studies
daily_c <- q("SELECT cancel_date AS d, start_status, COUNT(*) n FROM life
              WHERE cancel_date >= date_sub(DATE'{EVENT_FROM}', 7) AND cancel_date <= '{TO}' GROUP BY 1,2") %>% reg("daily_c")

uplift <- function(ev, keys, val) {
  ev %>% group_by(across(all_of(c(keys, "start_status")))) %>%
    summarise(pre_avg = mean(.data[[val]][offset_days < 0]), match_day = sum(.data[[val]][offset_days == 0]),
              post_avg = mean(.data[[val]][offset_days > 0]), .groups = "drop") %>%
    mutate(uplift_vs_pre_pct = round(100 * (match_day / pre_avg - 1), 1), across(c(pre_avg, match_day, post_avg), ~ round(.x, 0)))
}

# Top 10 competitions by viewers (World Cup analysed separately)
top_comp <- q("SELECT competitionName, COUNT(DISTINCT match_date) AS match_days, SUM(viewers) AS viewers FROM fixtures
               WHERE match_date >= '{EVENT_FROM}' AND match_date <= date_sub(DATE'{TO}', 7) AND competitionName IS NOT NULL
                 AND competitionName <> '' AND competitionName <> 'FIFA World Cup'
               GROUP BY 1 ORDER BY viewers DESC LIMIT 10") %>% collect()
sdf_copy_to(sc, top_comp, "top_comp", overwrite = TRUE)
ev_comp <- q("
  WITH f AS (SELECT DISTINCT x.competitionName, x.match_date FROM fixtures x JOIN top_comp t USING (competitionName)
             WHERE x.match_date >= '{EVENT_FROM}' AND x.match_date <= date_sub(DATE'{TO}', 7)),
  fn AS (SELECT *, COUNT(*) OVER (PARTITION BY competitionName) AS n_days FROM f)
  SELECT fn.competitionName, c.start_status, datediff(c.d, fn.match_date) AS offset_days, SUM(c.n) / MAX(fn.n_days) AS avg_cancels_per_match_day
  FROM fn JOIN daily_c c ON datediff(c.d, fn.match_date) BETWEEN -3 AND 7 GROUP BY 1,2,3") %>% collect()
comp_tbl <- uplift(ev_comp, "competitionName", "avg_cancels_per_match_day") %>% inner_join(top_comp, by = "competitionName") %>%
  arrange(desc(viewers), start_status) %>%
  dplyr::select(competitionName, match_days, viewers, start_status, pre_avg, match_day, post_avg, uplift_vs_pre_pct)
save_tbl(comp_tbl, "comp_tbl")

top_matches_sql <- function(where) paste0("
  SELECT match_opid, competitionName, home_team, away_team, to_date(match_date) AS match_date, COUNT(DISTINCT viewerId) AS viewers
  FROM {VIEW_TBL} WHERE ContentType2 = 'Football' AND match_opid IS NOT NULL AND ", where, "
    AND to_date(match_date) >= '{EVENT_FROM}' AND to_date(match_date) <= date_sub(DATE'{TO}', 7)
  GROUP BY 1,2,3,4,5 ORDER BY viewers DESC LIMIT 10")
match_event <- function(tm, view_name) {
  sdf_copy_to(sc, tm %>% mutate(match_date = as.character(match_date)) %>% dplyr::select(match_opid, match_date), view_name, overwrite = TRUE)
  ev <- q(paste0("SELECT m.match_opid, c.start_status, datediff(c.d, to_date(m.match_date)) AS offset_days, SUM(c.n) AS cancels
                  FROM ", view_name, " m JOIN daily_c c ON datediff(c.d, to_date(m.match_date)) BETWEEN -3 AND 7 GROUP BY 1,2,3")) %>% collect()
  uplift(ev, "match_opid", "cancels") %>% inner_join(tm, by = "match_opid") %>% arrange(desc(viewers), start_status)
}
top_matches <- q(top_matches_sql("competitionName <> 'FIFA World Cup'")) %>% collect()
match_tbl <- match_event(top_matches, "top_matches") %>%
  dplyr::select(match_date, competitionName, home_team, away_team, viewers, start_status, pre_avg, match_day, post_avg, uplift_vs_pre_pct)
save_tbl(match_tbl, "match_tbl")

# FIFA World Cup on its own (skipped automatically if there are no World Cup match days in the data)
wc_days <- q("SELECT match_date, SUM(viewers) AS viewers FROM fixtures
              WHERE competitionName = 'FIFA World Cup' AND match_date >= '{EVENT_FROM}' AND match_date <= '{TO}' GROUP BY 1
              HAVING SUM(viewers) >= 10000") %>% collect() %>% mutate(match_date = as.Date(match_date))   # drops stray low-viewer rows
if (nrow(wc_days) > 0) {
  wc_from <- min(wc_days$match_date); wc_to <- max(wc_days$match_date)
  cat(sprintf("FIFA World Cup: %d match days, %s to %s\n", nrow(wc_days), wc_from, wc_to))
  save_kv(list(match_days = nrow(wc_days), wc_from = as.character(wc_from), wc_to = as.character(wc_to)), "wc_meta")
  wc_daily <- q(sprintf("SELECT cancel_date AS d, start_status, COUNT(*) n FROM life WHERE cancel_date >= '%s' AND cancel_date <= '%s' GROUP BY 1,2",
                        wc_from - 28, wc_to + 28)) %>% collect() %>% mutate(d = as.Date(d))
  wc_period <- wc_daily %>% mutate(period = ifelse(d < wc_from, "1 Before", ifelse(d <= wc_to, "2 During", "3 After"))) %>%
    group_by(start_status, period) %>% summarise(avg_daily_cancels = round(mean(n), 0), .groups = "drop")
  save_tbl(wc_period, "wc_period")
  wc_matches <- q(top_matches_sql("competitionName = 'FIFA World Cup'")) %>% collect()
  wc_match_tbl <- match_event(wc_matches, "wc_matches") %>%
    dplyr::select(match_date, home_team, away_team, viewers, start_status, pre_avg, match_day, post_avg, uplift_vs_pre_pct)
  save_tbl(wc_match_tbl, "wc_match_tbl")
} else cat("No FIFA World Cup match days in the data: World Cup slide will be skipped\n")

In [0]:
save_tbl(q("
  SELECT CASE WHEN n_subs >= 6 THEN '6+' WHEN n_subs >= 4 THEN '4-5' ELSE CAST(n_subs AS STRING) END AS bucket, COUNT(*) AS customers
  FROM (SELECT Customer_External_ID, COUNT(*) n_subs FROM life
        WHERE Customer_External_ID IN (SELECT Customer_External_ID FROM cohort WHERE start_status = 'winback') GROUP BY 1)
  GROUP BY 1"), "winback_cycles")
display(q("SELECT n_ext, COUNT(*) customer_ids FROM (SELECT Customer_ID, COUNT(DISTINCT Customer_External_ID) n_ext FROM life GROUP BY 1) GROUP BY 1 ORDER BY 1"))
display(q("SELECT start_status, COUNT(*) subs, SUM(CASE WHEN gap_since_prev_paid_through < 0 THEN 1 ELSE 0 END) started_while_still_paid_up FROM cohort GROUP BY 1"))

In [0]:
reversed <- q("SELECT DISTINCT a.Subscription_ID FROM base_rows a JOIN base_rows b ON a.Subscription_ID = b.Subscription_ID
               WHERE a.cancel_date IS NOT NULL AND b.Subscription_version > a.Subscription_version AND b.cancel_date IS NULL") %>% reg("reversed")
renewed <- if (HAS_TXN) {
  q("SELECT DISTINCT c.Subscription_ID FROM cohort c JOIN {TXN_TBL} t ON t.Subscription_ID = c.Subscription_ID
     WHERE to_date(t.Transaction_Date) > c.cancel_date AND t.Payment_type = 'recurring'") %>% reg("renewed")
} else q("SELECT Subscription_ID FROM cohort WHERE 1 = 0") %>% reg("renewed")

post <- q("
  SELECT c.*, CASE WHEN expiry_date <= '{ASOF}' THEN 1 ELSE 0 END reached_expiry,
         CASE WHEN date_add(expiry_date, {MATURE_DAYS}) <= '{ASOF}' THEN 1 ELSE 0 END matured,
         CASE WHEN r.Subscription_ID IS NOT NULL THEN 1 ELSE 0 END cancel_reversed,
         CASE WHEN n.Subscription_ID IS NOT NULL THEN 1 ELSE 0 END charged_after_cancel
  FROM cohort c LEFT JOIN reversed r USING (Subscription_ID) LEFT JOIN renewed n USING (Subscription_ID)") %>% reg("post")
tbl_cache(sc, "post")

save_tbl(q("SELECT start_status, COUNT(*) AS subs, SUM(reached_expiry) AS reached_expiry, SUM(cancel_reversed) AS reversed,
                   SUM(charged_after_cancel) AS charged_after_cancel, SUM(CASE WHEN next_start <= expiry_date THEN 1 ELSE 0 END) AS new_sub_before_expiry
            FROM post GROUP BY 1"), "post_summary")

# One outcome per customer between cancelling and paid expiry (first match wins)
save_tbl(q("
  WITH s AS (
    SELECT p.Customer_External_ID, p.start_status, p.cancel_date, p.Subscription_ID, p.reached_expiry,
           CASE WHEN p.next_start <= p.expiry_date THEN 1 ELSE 0 END AS new_sub_before_expiry,
           CASE WHEN t.sessions_cancel_to_expiry > 0 THEN 1 ELSE 0 END AS kept_watching
    FROM post p JOIN tiers t ON t.Subscription_ID = p.Subscription_ID),
  c AS (
    SELECT Customer_External_ID, MAX_BY(start_status, struct(cancel_date, Subscription_ID)) AS start_status,
           MAX(new_sub_before_expiry) AS new_sub, MIN(reached_expiry) AS all_expired, MAX(kept_watching) AS kept_watching
    FROM s GROUP BY 1)
  SELECT start_status,
         CASE WHEN new_sub = 1 THEN '1 Started a new subscription before expiry'
              WHEN all_expired = 0 THEN '2 Still in paid period'
              WHEN kept_watching = 1 THEN '3 Kept watching, then expired'
              ELSE '4 Stopped watching, then expired' END AS outcome,
         COUNT(*) AS customers
  FROM c GROUP BY 1,2"), "outcomes")

ret <- q("
  SELECT start_status, COUNT(*) AS matured_subs,
         SUM(CASE WHEN days_expiry_to_next_start >= 0 THEN 1 ELSE 0 END) AS returned,
         SUM(CASE WHEN days_expiry_to_next_start BETWEEN 0 AND 7  THEN 1 ELSE 0 END) AS d0_7,
         SUM(CASE WHEN days_expiry_to_next_start BETWEEN 8 AND 14 THEN 1 ELSE 0 END) AS d8_14,
         SUM(CASE WHEN days_expiry_to_next_start BETWEEN 15 AND 30 THEN 1 ELSE 0 END) AS d15_30
  FROM post WHERE matured = 1 GROUP BY 1") %>% collect() %>%
  mutate(later = returned - (d0_7 + d8_14 + d15_30), not_back = matured_subs - returned)
save_tbl(ret, "return_speed")
save_tbl(q("SELECT start_status, next_channel, COUNT(*) AS returns FROM post WHERE days_expiry_to_next_start >= 0 GROUP BY 1,2 ORDER BY returns DESC"), "returns_channel")
save_tbl(q("
  SELECT p.start_status, v.competitionName, COUNT(DISTINCT p.Customer_External_ID) AS returners, ROUND(SUM(v.minutes),0) AS minutes
  FROM post p JOIN v ON v.viewerId = p.Customer_External_ID AND v.vdate BETWEEN p.next_start AND date_add(p.next_start, 7)
  WHERE p.days_expiry_to_next_start >= 0 GROUP BY 1,2 HAVING COUNT(DISTINCT p.Customer_External_ID) >= 20 ORDER BY minutes DESC"), "returners_comp")

In [0]:
saved <- list.files(DATA_DIR, pattern = "\\.csv$")
need  <- c("params", "kpi", "monthly", "never_watched", "tiers", "nw_product_country", "event_meta", "ev_study", "comp_tbl",
           "match_tbl", "outcomes", "post_summary", "returns_channel", "returners_comp")
print(data.frame(table = sub("\\.csv$", "", saved), size_kb = round(file.size(file.path(DATA_DIR, saved)) / 1024, 1)))
missing <- setdiff(need, sub("\\.csv$", "", saved))
if (length(missing)) stop("Required tables missing: ", paste(missing, collapse = ", ")) else cat("All required tables saved to", DATA_DIR, "\n")

In [0]:
G_DK <- "#6B7280"; G_LT <- "#C9CED8"; G_MID <- "#9AA1AF"; G_INK <- "#222B3A"; G_MUT <- "#5B6577"; G_BG <- "#FAFBFD"; G_GRID <- "#E3E7EE"
G6 <- c("#E6E9EF", "#D0D5DF", "#B3BAC7", "#8E96A6", "#6B7280", "#3F4756")
STATUS_COL <- c("New" = G_LT, "Win-back" = G_DK)
SIZE <- list(c1 = c(8.6, 4.7), c2 = c(8.6, 4.7), c3 = c(8.3, 3.4), c4 = c(8.3, 3.4), c5 = c(8.3, 4.6), c5b = c(8.3, 4.6),
             c6 = c(8.3, 4.6), c7 = c(8.3, 4.35), c8 = c(8.3, 4.35), c9 = c(6.0, 4.4), c10 = c(6.4, 4.4), c11 = c(8.3, 3.6),
             c12 = c(5.9, 4.4), c13 = c(6.4, 4.4), c14 = c(8.3, 4.4), c15 = c(8.3, 4.4), c16 = c(8.3, 3.4))   # same as the slide boxes
if (P$CHART_SOURCE == "ggplot2") unlink(list.files(CHART_DIR, pattern = "\\.png$", full.names = TRUE))

lab_status <- function(x) factor(ifelse(x == "new", "New", "Win-back"), levels = c("New", "Win-back"))
theme_tod <- function() theme_minimal(base_size = 11) + theme(
  plot.background = element_rect(fill = G_BG, colour = NA), panel.background = element_rect(fill = G_BG, colour = NA),
  panel.grid = element_blank(), axis.text = element_text(colour = G_MUT), axis.title = element_text(colour = G_MUT),
  legend.position = "top", legend.justification = "left", legend.title = element_blank(), legend.text = element_text(colour = G_MUT))
rd <- function(n) { f <- file.path(DATA_DIR, paste0(n, ".csv")); if (file.exists(f)) read.csv(f, stringsAsFactors = FALSE, check.names = FALSE) else NULL }
gsave <- function(p, name) {
  print(p)
  if (P$CHART_SOURCE == "ggplot2")
    ggsave(file.path(CHART_DIR, paste0(name, ".png")), p, width = SIZE[[name]][1], height = SIZE[[name]][2], dpi = 200, bg = G_BG)
}
win_from <- substr(P$FROM, 1, 7)

stacked_months <- function(df, value, name, money = FALSE) {
  d <- df %>% mutate(status = lab_status(start_status), v = .data[[value]], win = m >= win_from)
  tt <- d %>% group_by(m, win) %>% summarise(t = sum(v), .groups = "drop")
  avg <- tt %>% filter(!win) %>% summarise(a = mean(t)) %>% pull(a)
  p <- ggplot(d, aes(m, v, fill = status, alpha = win)) + geom_col(width = 0.75) +
    geom_text(data = tt, aes(m, t, label = if (money) paste0("$", comma(round(t / 1000)), "K") else paste0(round(t / 1000), "K")),
              inherit.aes = FALSE, vjust = -0.4, size = 2.5, colour = G_INK) +
    scale_fill_manual(values = STATUS_COL, labels = c("Started as New", "Started as Win-back")) +
    scale_alpha_manual(values = c(`FALSE` = 0.55, `TRUE` = 1), guide = "none") +
    scale_y_continuous(expand = expansion(mult = c(0, 0.12))) + labs(x = NULL, y = NULL) + theme_tod() +
    theme(axis.text.y = element_blank(), axis.text.x = element_text(angle = 45, hjust = 1))
  if (!money && length(avg) && !is.na(avg)) p <- p + geom_hline(yintercept = avg, linetype = "dashed", colour = G_INK) +
    annotate("text", x = 1, y = avg, label = paste0("History average ", round(avg / 1000, 1), "K"), hjust = 0, vjust = -0.5, size = 3, colour = G_INK)
  gsave(p, name)
}
stack100 <- function(d, cat_col, levels_, colours, name, counts = FALSE, min_share = 0.05, labels_ = sub("^[0-9] ", "", levels_)) {
  d <- d %>% mutate(cat = factor(.data[[cat_col]], levels = levels_)) %>% group_by(row) %>% mutate(share = n / sum(n)) %>% ungroup()
  d$lbl <- if (counts) paste0(percent(d$share, accuracy = 1), "\n", comma(d$n)) else percent(d$share, accuracy = 1)
  dark <- setNames(as.integer(grDevices::col2rgb(colours) %>% { 0.299 * .[1, ] + 0.587 * .[2, ] + 0.114 * .[3, ] } < 150), levels_)
  p <- ggplot(d, aes(share, row, fill = cat)) + geom_col(width = 0.55, position = position_stack(reverse = TRUE), colour = G_BG) +
    geom_text(data = filter(d, share >= min_share), aes(label = lbl, colour = factor(dark[as.character(cat)])),
              position = position_stack(vjust = 0.5, reverse = TRUE), size = 3, fontface = "bold", show.legend = FALSE) +
    scale_colour_manual(values = c(`1` = "white", `0` = G_INK), guide = "none") +
    scale_fill_manual(values = setNames(colours, levels_), labels = labels_) +
    scale_x_continuous(expand = c(0, 0)) + labs(x = NULL, y = NULL) + theme_tod() +
    theme(axis.text.x = element_blank(), legend.position = "bottom", legend.justification = "center")
  gsave(p, name)
}
hgroup <- function(d, lab_col, val_col, order, name, fmt = function(v) sprintf("%+.0f%%", v)) {
  d <- d %>% mutate(lab = factor(.data[[lab_col]], levels = rev(order)), status = factor(lab_status(start_status), levels = c("Win-back", "New")),
                    v = .data[[val_col]]) %>% filter(!is.na(lab))
  p <- ggplot(d, aes(v, lab, fill = status)) + geom_col(position = position_dodge(width = 0.8), width = 0.75) +
    geom_text(aes(label = fmt(v), hjust = ifelse(v >= 0, -0.15, 1.15)), position = position_dodge(width = 0.8), size = 2.6, colour = G_INK) +
    scale_fill_manual(values = STATUS_COL, breaks = c("New", "Win-back")) +
    scale_x_continuous(expand = expansion(mult = c(0.15, 0.2))) + labs(x = NULL, y = NULL) + theme_tod() +
    theme(axis.text.x = element_blank(), legend.position = "bottom", legend.justification = "right")
  if (min(d$v, na.rm = TRUE) < 0) p <- p + geom_vline(xintercept = 0, colour = G_MID)
  gsave(p, name)
}

stacked_months(rd("monthly"), "subs", "c1")
if (!is.null(rd("rev"))) stacked_months(rd("rev"), "renewal_value_forgone_usd", "c2", money = TRUE)

ti <- rd("tiers"); lv <- sort(unique(ti$mou_tier))
stack100(ti %>% group_by(row = lab_status(start_status), mou_tier) %>% summarise(n = sum(subs), .groups = "drop") %>%
           mutate(row = factor(row, levels = c("Win-back", "New"))), "mou_tier", lv, tail(G6, length(lv)), "c3")
att_lv <- c("watched this subscription", "watched only on another subscription / pass", "did not watch this subscription")
stack100(ti %>% group_by(row = lab_status(start_status), attribution) %>% summarise(n = sum(subs), .groups = "drop") %>%
           mutate(row = factor(row, levels = c("Win-back", "New"))), "attribution", att_lv, c(G_DK, G_MID, G_LT), "c4",
         labels_ = c("Watched on this subscription", "Watched only on another subscription / pass", "Did not watch"))

bc <- rd("nw_product_country") %>% group_by(Country) %>% summarise(subs = sum(subs), .groups = "drop") %>% arrange(desc(subs)) %>% head(7)
gsave(ggplot(bc, aes(subs, factor(Country, levels = rev(Country)))) + geom_col(fill = G_DK, width = 0.62) +
        geom_text(aes(label = comma(subs)), hjust = -0.15, size = 3.2, fontface = "bold", colour = G_INK) +
        scale_x_continuous(expand = expansion(mult = c(0, 0.15))) + labs(x = NULL, y = NULL) + theme_tod() + theme(axis.text.x = element_blank()), "c5")

np <- rd("nw_problems")
if (!is.null(np)) {
  sig <- c(never_opened_player = "Never opened the player", had_error_coded_session = "Had an error-coded session",
           had_entitlement_error = "Had an entitlement error", opened_no_error_no_play = "Opened, no error, no play")
  npl <- np %>% pivot_longer(all_of(names(sig)), names_to = "signal", values_to = "k") %>% mutate(share = 100 * k / non_watching_subs, signal = sig[signal])
  hgroup(npl, "signal", "share", unname(sig), "c5b", fmt = function(v) sprintf("%.0f%%", v))
}

ev <- rd("ev_study") %>% mutate(status = lab_status(start_status))
gsave(ggplot(ev, aes(offset_days, avg_cancels_per_peak_day, colour = status)) + geom_vline(xintercept = 0, linetype = "dashed", colour = G_INK) +
        geom_line(linewidth = 1.1) + geom_point(size = 2) + scale_colour_manual(values = c(New = G_MID, `Win-back` = G_DK)) +
        scale_x_continuous(breaks = -3:7) + scale_y_continuous(limits = c(0, NA), labels = comma) +
        labs(x = "Days relative to peak match day", y = "Cancellations per match day") + theme_tod() +
        theme(panel.grid.major.y = element_line(colour = G_GRID), legend.position = "bottom", legend.justification = "right"), "c6")

ct <- rd("comp_tbl"); hgroup(ct, "competitionName", "uplift_vs_pre_pct", ct %>% distinct(competitionName, viewers) %>% arrange(desc(viewers)) %>% pull(competitionName), "c7")
mlab <- function(d) d %>% mutate(label = paste0(home_team, " v ", away_team, " (", format(as.Date(match_date), "%d %b"), ")"))
mt <- mlab(rd("match_tbl")); hgroup(mt, "label", "uplift_vs_pre_pct", mt %>% distinct(label, viewers) %>% arrange(desc(viewers)) %>% pull(label), "c8")

wp <- rd("wc_period")
if (!is.null(wp)) {
  wpd <- wp %>% mutate(status = lab_status(start_status), period = factor(period, labels = c("4 weeks before", "During", "4 weeks after")))
  gsave(ggplot(wpd, aes(period, avg_daily_cancels, fill = status)) + geom_col(position = position_dodge(width = 0.8), width = 0.75) +
          geom_text(aes(label = comma(avg_daily_cancels)), position = position_dodge(width = 0.8), vjust = -0.4, size = 3.2, fontface = "bold", colour = G_INK) +
          scale_fill_manual(values = STATUS_COL) + scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
          labs(x = NULL, y = NULL) + theme_tod() + theme(axis.text.y = element_blank()), "c9")
  wcm <- rd("wc_match_tbl"); wc_to <- rd("wc_meta") %>% filter(key == "wc_to") %>% pull(value)
  wcm <- wcm %>% mutate(label = paste0(home_team, " v ", away_team, ifelse(as.character(match_date) == wc_to, " (final)", "")))
  hgroup(wcm, "label", "uplift_vs_pre_pct", wcm %>% distinct(label, viewers) %>% arrange(desc(viewers)) %>% pull(label), "c10")
}

oc <- rd("outcomes"); oc_lv <- sort(unique(oc$outcome)); ocn <- oc %>% group_by(start_status) %>% summarise(t = sum(customers))
oc <- oc %>% left_join(ocn, by = "start_status") %>%
  mutate(row = factor(paste0(as.character(lab_status(start_status)), "\n(", comma(t), " customers)")), n = customers)
oc$row <- factor(oc$row, levels = rev(sort(unique(as.character(oc$row)))))
stack100(oc, "outcome", oc_lv, c("#3F4756", G_MID, "#8E96A6", G_LT), "c11", counts = TRUE, min_share = 0.08)

rc <- rd("returns_channel")
hgroup(rc, "next_channel", "returns", rc %>% group_by(next_channel) %>% summarise(t = sum(returns)) %>% arrange(desc(t)) %>% pull(next_channel), "c12",
       fmt = function(v) comma(v))
rcp <- rd("returners_comp") %>% filter(!is.na(competitionName), competitionName != "", competitionName != "NA")
top7 <- rcp %>% group_by(competitionName) %>% summarise(t = sum(returners)) %>% arrange(desc(t)) %>% head(7) %>% pull(competitionName)
hgroup(rcp %>% filter(competitionName %in% top7), "competitionName", "returners", top7, "c13", fmt = function(v) comma(v))

cs <- rd("cancel_speed")
if (!is.null(cs)) {
  csl <- cs %>% pivot_longer(c(within_1d, within_7d, within_30d), names_to = "when", values_to = "share") %>%
    mutate(series = factor(paste(as.character(lab_status(start_status)), ifelse(period == "History", "history", "review window"), sep = ", "),
                           levels = c("New, history", "New, review window", "Win-back, history", "Win-back, review window")),
           when = factor(when, levels = c("within_1d", "within_7d", "within_30d"), labels = c("Within 1 day", "Within 7 days", "Within 30 days")))
  gsave(ggplot(csl, aes(when, share, fill = series)) + geom_col(position = position_dodge(width = 0.85), width = 0.8) +
          geom_text(aes(label = percent(share, accuracy = 1)), position = position_dodge(width = 0.85), vjust = -0.4, size = 2.7, colour = G_INK) +
          scale_fill_manual(values = c("#E6E9EF", G_LT, G_MID, G_DK)) + scale_y_continuous(limits = c(0, 1.12), expand = c(0, 0)) +
          labs(x = NULL, y = NULL) + theme_tod() + theme(axis.text.y = element_blank()), "c14")
}

cy <- rd("winback_cycles")
if (!is.null(cy)) {
  cy <- cy %>% mutate(bucket = factor(as.character(bucket), levels = c("1", "2", "3", "4-5", "6+")), share = customers / sum(customers))
  gsave(ggplot(cy, aes(bucket, customers)) + geom_col(fill = G_DK, width = 0.6) +
          geom_text(aes(label = paste0(comma(customers), "\n", percent(share, accuracy = 1))), vjust = -0.3, size = 3.2, fontface = "bold", colour = G_INK) +
          scale_x_discrete(labels = c("1*", "2", "3", "4-5", "6+")) + scale_y_continuous(expand = expansion(mult = c(0, 0.22))) +
          labs(x = "Lifetime D2C subscriptions per customer", y = NULL) + theme_tod() + theme(axis.text.y = element_blank()), "c15")
}

rs <- rd("return_speed")
if (!is.null(rs)) {
  rs_lv <- c("d0_7", "d8_14", "d15_30", "later", "not_back")
  rsl <- rs %>% pivot_longer(all_of(rs_lv), names_to = "when", values_to = "n") %>%
    mutate(row = factor(as.character(lab_status(start_status)), levels = c("Win-back", "New")))
  stack100(rsl, "when", rs_lv, c("#3F4756", G_DK, "#8E96A6", "#B3BAC7", "#E6E9EF"), "c16",
           labels_ = c("Back within 7 days", "8-14 days", "15-30 days", "More than 30 days", "Not back (yet)"))
}
cat("ggplot2 charts done", if (P$CHART_SOURCE == "ggplot2") paste("and saved to", CHART_DIR) else "(display only; the PPT uses the Python charts)", "\n")

## Part 2: charts and PowerPoint in Python (matplotlib + python-pptx)
Reads the CSV tables saved by Part 1 from `OUT_ROOT/data`.

In [0]:
%python
# python-pptx builds the PowerPoint; matplotlib draws the charts (both run on the driver).
# If your cluster blocks pip, add python-pptx as a cluster library instead and delete these two lines.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-pptx"], check=False)

import os, math, datetime as dt
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.oxml.ns import qn
print("python-pptx ready")

In [0]:
%python
try:
    OUT_ROOT = dbutils.widgets.get("OUT_ROOT")          # same widget the R cells use
except NameError:                                     # running outside Databricks
    OUT_ROOT = os.environ.get("OUT_ROOT", "/dbfs/FileStore/cancellation_ppt")
DATA_DIR, CHART_DIR = os.path.join(OUT_ROOT, "data"), os.path.join(OUT_ROOT, "charts")
os.makedirs(CHART_DIR, exist_ok=True)

def read(name):
    """Read one table saved by R; returns None when the table was not produced (for example no transactions table)."""
    p = os.path.join(DATA_DIR, name + ".csv")
    if not os.path.exists(p) or os.path.getsize(p) == 0:
        print(f"  (no {name}.csv - slides that need it will be skipped)")
        return None
    return pd.read_csv(p)

prm = read("params")
P = dict(zip(prm["key"], prm["value"].astype(str)))
T = {n: read(n) for n in ["kpi", "monthly", "rev", "never_watched", "tiers", "nw_problems", "nw_product_country",
                          "event_meta", "ev_study", "comp_tbl", "match_tbl", "wc_meta", "wc_period", "wc_match_tbl",
                          "cancel_speed", "winback_cycles", "post_summary", "outcomes", "return_speed",
                          "returns_channel", "returners_comp"]}
print("Parameters:", {k: P[k] for k in ["FROM", "TO", "HIST_FROM", "EVENT_FROM", "ASOF", "CHART_SOURCE"] if k in P})

In [0]:
%python
D = lambda s: dt.date.fromisoformat(str(s)[:10])
FROM, TO, ASOF, HIST_FROM = D(P["FROM"]), D(P["TO"]), D(P["ASOF"]), D(P["HIST_FROM"])
mon = lambda d: d.strftime("%b")
WIN = (f"{mon(FROM)} {FROM.year}" if (FROM.year, FROM.month) == (TO.year, TO.month)
       else f"{mon(FROM)}-{mon(TO)} {TO.year}" if FROM.year == TO.year else f"{mon(FROM)} {FROM.year}-{mon(TO)} {TO.year}")
ASOF_TXT = f"{ASOF.day} {ASOF.strftime('%b %Y')}"
LAB = {"new": "New", "winback": "Win-back"}

def k(v, d=1):   return f"{v/1000:,.{d}f}K"
def n0(v):       return f"{v:,.0f}"
def pc(v, d=0):  return f"{100*v:.{d}f}%"
def sp(v, d=0):  return f"{v:+.{d}f}%"
def usd(v):      return f"${v/1e6:,.2f}M" if v >= 1e6 else f"${v/1000:,.0f}K"
def mlabel(m):   return dt.date.fromisoformat(m + "-01").strftime("%b %Y")

M = {}
kp = T["kpi"].set_index("start_status")
M["subs"], M["cust"] = kp.loc["all", "subs"], kp.loc["all", "customers"]
M["subs_new"], M["subs_wb"] = kp.loc["new", "subs"], kp.loc["winback", "subs"]
M["wb_share"] = M["subs_wb"] / M["subs"]

mo = T["monthly"].copy()
subs_m = mo.pivot_table(index="m", columns="start_status", values="subs", fill_value=0).sort_index()
subs_m["total"] = subs_m.sum(axis=1)
cust_m = mo.groupby("m")["customers"].sum().sort_index()
win_months = [m for m in subs_m.index if FROM.strftime("%Y-%m") <= m <= TO.strftime("%Y-%m")]
hist_months = [m for m in subs_m.index if m < FROM.strftime("%Y-%m")]
M["win_months"], M["hist_label"] = win_months, f"{mlabel(hist_months[0])}-{mlabel(hist_months[-1])}" if hist_months else ""
M["hist_avg"] = subs_m.loc[hist_months, "total"].mean() if hist_months else np.nan
M["last_m"] = win_months[-1]
M["last_v"] = subs_m.loc[M["last_m"], "total"]
M["last_vs_avg"] = M["last_v"] / M["hist_avg"] - 1
M["peak_m"] = subs_m["total"].idxmax(); M["peak_v"] = subs_m["total"].max()
wbs = subs_m.get("winback", 0) / subs_m["total"]
M["wb_share_win"] = (wbs[win_months].min(), wbs[win_months].max())
M["wb_share_hist"] = (wbs[hist_months].min(), wbs[hist_months].max()) if hist_months else (np.nan, np.nan)
if len(hist_months) > 1:
    c_hist = cust_m[hist_months]
    M["cust_mean"], M["cust_sd"] = c_hist.mean(), c_hist.std()
    M["cust_last"] = cust_m[M["last_m"]]
    M["z"] = (M["cust_last"] - M["cust_mean"]) / M["cust_sd"]

if T["rev"] is not None:
    rv = T["rev"].pivot_table(index="m", columns="start_status", values=["renewal_value_forgone_usd", "subs"], aggfunc="sum", fill_value=0).sort_index()
    val = rv["renewal_value_forgone_usd"].sum(axis=1); rsubs = rv["subs"].sum(axis=1)
    M["rev_win"] = val[[m for m in win_months if m in val.index]].sum()
    M["rev_by_month"] = {m: val.get(m, 0) for m in win_months}
    M["avg_charge_win"] = M["rev_win"] / rsubs[[m for m in win_months if m in rsubs.index]].sum()
    hm = [m for m in hist_months if m in val.index]
    M["avg_charge_hist"] = val[hm].sum() / rsubs[hm].sum() if hm else np.nan
    M["rev_peak_m"] = val.idxmax(); M["rev_peak_v"] = val.max()

nwt = T["never_watched"].set_index("start_status")
M["never"] = nwt["never"].sum(); M["never_share"] = M["never"] / nwt["subs"].sum()
M["never_new"], M["never_wb"] = nwt.loc["new", "never"], nwt.loc["winback", "never"]
M["never_new_sh"], M["never_wb_sh"] = M["never_new"] / nwt.loc["new", "subs"], M["never_wb"] / nwt.loc["winback", "subs"]

ti = T["tiers"]
tier_tot = ti.groupby("start_status")["subs"].sum()
M["heavy_sh"] = (ti[ti.mou_tier.str.startswith("5")].groupby("start_status")["subs"].sum() / tier_tot).to_dict()
att = ti.groupby(["start_status", "attribution"])["subs"].sum().unstack(fill_value=0)
att_sh = att.div(att.sum(axis=1), axis=0)
OTHER, OWN, NONE = "watched only on another subscription / pass", "watched this subscription", "did not watch this subscription"
M["other_wb"], M["other_new"] = att_sh.loc["winback"].get(OTHER, 0), att_sh.loc["new"].get(OTHER, 0)
M["other_wb_n"], M["other_new_n"] = att.loc["winback"].get(OTHER, 0), att.loc["new"].get(OTHER, 0)
M["own_wb"], M["own_new"] = att_sh.loc["winback"].get(OWN, 0), att_sh.loc["new"].get(OWN, 0)

pcn = T["nw_product_country"]
by_c = pcn.groupby("Country")["subs"].sum().sort_values(ascending=False)
M["nw_total"] = pcn["subs"].sum()
M["top_c"], M["top_c_v"] = by_c.index[0], by_c.iloc[0]
M["top3_c"], M["top3_sh"] = list(by_c.index[:3]), by_c.iloc[:3].sum() / M["nw_total"]
cells = pcn.groupby(["Product_name", "Country"])["subs"].sum().sort_values(ascending=False)
M["top_cells"] = [(p, c, v) for (p, c), v in cells.head(3).items()]
top_c_products = pcn[pcn.Country.isin(by_c.index[:2])].groupby("Product_name")["subs"].sum().sort_values(ascending=False)
M["top_products"] = list(top_c_products.index[:3])

em = dict(zip(T["event_meta"]["key"], T["event_meta"]["value"]))
ev = T["ev_study"]
d0 = ev[ev.offset_days == 0].set_index("start_status")["avg_cancels_per_peak_day"]
M["d0_new"], M["d0_wb"] = d0.get("new", np.nan), d0.get("winback", np.nan)
M["avg_new"], M["avg_wb"] = float(em["avg_daily_new"]), float(em["avg_daily_winback"])
M["ev_up_new"], M["ev_up_wb"] = M["d0_new"] / M["avg_new"] - 1, M["d0_wb"] / M["avg_wb"] - 1
post = ev[ev.offset_days.between(1, 7)].groupby("start_status")["avg_cancels_per_peak_day"].mean()
M["post_new_vs_avg"] = post.get("new", np.nan) / M["avg_new"] - 1
M["peak_days"], M["match_days"] = int(float(em["peak_days"])), int(float(em["match_days"]))
EV_FROM = D(P["EVENT_FROM"])

ct = T["comp_tbl"]
ct_new = ct[ct.start_status == "new"].sort_values("uplift_vs_pre_pct", ascending=False)
ct_wb = ct[ct.start_status == "winback"].sort_values("uplift_vs_pre_pct", ascending=False)
M["comp_top"], M["comp_top_up"] = ct_new.iloc[0]["competitionName"], ct_new.iloc[0]["uplift_vs_pre_pct"]
M["comp_top_days"] = int(ct_new.iloc[0]["match_days"])
M["comp_top_wb_up"] = ct_wb.set_index("competitionName")["uplift_vs_pre_pct"].get(M["comp_top"], np.nan)
M["comp_wb_top"], M["comp_wb_top_up"] = ct_wb.iloc[0]["competitionName"], ct_wb.iloc[0]["uplift_vs_pre_pct"]
busiest = ct.drop_duplicates("competitionName").sort_values("match_days", ascending=False).head(2)["competitionName"].tolist()
bus_up = ct[ct.competitionName.isin(busiest)]["uplift_vs_pre_pct"]
M["busiest"], M["bus_lo"], M["bus_hi"] = busiest, bus_up.min(), bus_up.max()

def match_label(r, comp=True):
    d = D(r["match_date"])
    return f"{r['home_team']} v {r['away_team']} ({d.day} {d.strftime('%b')})"
mt = T["match_tbl"]
mt_new = mt[mt.start_status == "new"].sort_values("uplift_vs_pre_pct", ascending=False).iloc[0]
mt_wb = mt[(mt.start_status == "winback") & (mt.match_date == mt_new["match_date"]) & (mt.home_team == mt_new["home_team"])].iloc[0]
M["match_top"], M["match_top_comp"] = match_label(mt_new), mt_new["competitionName"]
M["match_top_new"] = (mt_new["pre_avg"], mt_new["match_day"], mt_new["uplift_vs_pre_pct"])
M["match_top_wb"] = (mt_wb["pre_avg"], mt_wb["match_day"], mt_wb["uplift_vs_pre_pct"])
mt_comp = mt.drop_duplicates(["match_date", "home_team"])["competitionName"].value_counts()
M["match_main_comp"], M["match_main_n"], M["match_n"] = mt_comp.index[0], mt_comp.iloc[0], mt_comp.sum()

if T["wc_period"] is not None:
    wp = T["wc_period"].pivot_table(index="start_status", columns="period", values="avg_daily_cancels")
    M["wc_chg_new"] = wp.loc["new", "2 During"] / wp.loc["new", "1 Before"] - 1
    M["wc_chg_wb"] = wp.loc["winback", "2 During"] / wp.loc["winback", "1 Before"] - 1
    wm = dict(zip(T["wc_meta"]["key"], T["wc_meta"]["value"]))
    M["wc_days"], M["wc_from"], M["wc_to"] = int(float(wm["match_days"])), D(wm["wc_from"]), D(wm["wc_to"])
    wmt = T["wc_match_tbl"]
    fin = wmt[pd.to_datetime(wmt.match_date).dt.date == M["wc_to"]]
    M["wc_final"] = None
    if len(fin):
        fn = fin[fin.start_status == "new"].iloc[0]
        M["wc_final"] = (f"{fn['home_team']} v {fn['away_team']}", fn["uplift_vs_pre_pct"], fn["pre_avg"], fn["match_day"])

oc = T["outcomes"].copy()
oc["share"] = oc["customers"] / oc.groupby("start_status")["customers"].transform("sum")
osh = oc.pivot_table(index="start_status", columns="outcome", values="share", fill_value=0)
ocn = oc.groupby("start_status")["customers"].sum()
def oshare(st, prefix): return osh.loc[st, [c for c in osh.columns if c.startswith(prefix)]].sum()
M["oc_newsub"] = (oshare("new", "1"), oshare("winback", "1"))
M["oc_paid"] = (oshare("new", "2"), oshare("winback", "2"))
M["oc_kept"] = (oshare("new", "3"), oshare("winback", "3"))
M["oc_stop"] = (oshare("new", "4"), oshare("winback", "4"))
M["oc_n"] = (ocn.get("new", 0), ocn.get("winback", 0))
ps = T["post_summary"]
M["charged_after"], M["reversed"] = ps["charged_after_cancel"].sum(), ps["reversed"].sum()

rc = T["returns_channel"]
rc_wb = rc[rc.start_status == "winback"].sort_values("returns", ascending=False)
M["ret_wb"], M["ret_new"] = rc_wb["returns"].sum(), rc[rc.start_status == "new"]["returns"].sum()
M["ret_top_ch"], M["ret_top_sh"] = rc_wb.iloc[0]["next_channel"], rc_wb.iloc[0]["returns"] / M["ret_wb"]
rcp = T["returners_comp"]
rcp = rcp[rcp.competitionName.notna() & (rcp.competitionName.astype(str).str.strip() != "") & (rcp.competitionName != "NA")]
M["ret_comps"] = rcp.groupby("competitionName")["returners"].sum().sort_values(ascending=False).index[:3].tolist()

for key, v in M.items():
    if not isinstance(v, (list, dict, tuple)):
        print(f"{key:>18}: {v}")

In [0]:
%python
DK, LT, MID, INK, MUT, BG, GRID = "#6B7280", "#C9CED8", "#9AA1AF", "#222B3A", "#5B6577", "#FAFBFD", "#E3E7EE"
G6 = ["#E6E9EF", "#D0D5DF", "#B3BAC7", "#8E96A6", "#6B7280", "#3F4756"]
# Chart sizes in inches = the picture boxes on the slides, so nothing is stretched
SIZE = {"c1": (8.6, 4.7), "c2": (8.6, 4.7), "c3": (8.3, 3.4), "c4": (8.3, 3.4), "c5": (8.3, 4.6), "c5b": (8.3, 4.6),
        "c6": (8.3, 4.6), "c7": (8.3, 4.35), "c8": (8.3, 4.35), "c9": (6.0, 4.4), "c10": (6.4, 4.4), "c11": (8.3, 3.6),
        "c12": (5.9, 4.4), "c13": (6.4, 4.4), "c14": (8.3, 4.4), "c15": (8.3, 4.4), "c16": (8.3, 3.4)}
plt.rcParams["font.family"] = "DejaVu Sans"

def fig(name):
    w, h = SIZE[name]
    f = plt.figure(figsize=(w, h), dpi=200, facecolor=BG); a = f.add_subplot(111, facecolor=BG)
    for s in ("top", "right"): a.spines[s].set_visible(False)
    for s in ("left", "bottom"): a.spines[s].set_color(LT)
    a.tick_params(length=0, labelcolor=MUT); return f, a

def save(f, name):
    f.tight_layout(pad=0.6); f.savefig(os.path.join(CHART_DIR, name + ".png"), facecolor=BG); plt.close(f)

def txtcol(c):
    r, g, b = (int(c[i:i + 2], 16) for i in (1, 3, 5)); return "white" if 0.299 * r + 0.587 * g + 0.114 * b < 150 else INK

def short_m(m): return dt.date.fromisoformat(m + "-01").strftime("%b-%y")

def stacked_months(name, df, money=False, avg=None):
    f, ax = fig(name); x = np.arange(len(df)); win = [m in win_months for m in df.index]
    a, b = df.get("new", pd.Series(0, index=df.index)).values, df.get("winback", pd.Series(0, index=df.index)).values
    top = (a + b).max()
    for i in x:
        al = 1 if win[i] else 0.55
        ax.bar(i, b[i], color=DK, alpha=al, width=0.75); ax.bar(i, a[i], bottom=b[i], color=LT, alpha=al, width=0.75)
        lab = f"${(a[i]+b[i])/1000:,.0f}K" if money else f"{(a[i]+b[i])/1000:,.0f}K"
        ax.text(i, a[i] + b[i] + top * 0.01, lab, ha="center", va="bottom", fontsize=7.5, color=INK, fontweight="bold" if win[i] else "normal")
    if avg is not None and not np.isnan(avg):
        ax.axhline(avg, color=INK, ls="--", lw=1)
        ax.text(0, avg + top * 0.015, f"{M['hist_label']} average {avg/1000:,.1f}K", fontsize=9, color=INK, va="bottom")
    ax.set_xticks(x); ax.set_xticklabels([short_m(m) for m in df.index], rotation=45, ha="right", fontsize=8.5)
    ax.yaxis.set_visible(False); ax.spines["left"].set_visible(False); ax.set_ylim(0, top * 1.15)
    ax.legend(handles=[Patch(color=LT, label="Started as New"), Patch(color=DK, label="Started as Win-back")], frameon=False,
              fontsize=10, labelcolor=MUT, loc="upper left", ncol=2, bbox_to_anchor=(0, 1.03))
    save(f, name)

def stack100(name, cats, rows, labels, colors, fs=10, minshare=0.05, counts=False):
    f, ax = fig(name); y = np.arange(len(cats))[::-1]; left = np.zeros(len(cats))
    tot = np.array([sum(rows[c]) for c in cats], float)
    for j, (lab, col) in enumerate(zip(labels, colors)):
        vals = np.array([rows[c][j] for c in cats], float); sh = np.divide(vals, tot, out=np.zeros_like(vals), where=tot > 0)
        ax.barh(y, sh, left=left, color=col, height=0.55, label=lab, edgecolor=BG, linewidth=1.5)
        for i in range(len(cats)):
            if sh[i] >= minshare:
                t = f"{sh[i]*100:.0f}%" + (f"\n{int(vals[i]):,}" if counts else "")
                ax.text(left[i] + sh[i] / 2, y[i], t, ha="center", va="center", fontsize=fs, color=txtcol(col), fontweight="bold")
        left += sh
    ax.set_yticks(y); ax.set_yticklabels(cats, fontsize=11, color=INK); ax.set_xlim(0, 1)
    ax.xaxis.set_visible(False); ax.spines["bottom"].set_visible(False)
    ax.legend(frameon=False, fontsize=8.5, labelcolor=MUT, ncol=3, loc="upper center", bbox_to_anchor=(0.45, -0.02)); save(f, name)

def hbar(name, labels, vals, fmt="{:,.0f}"):
    f, ax = fig(name); y = np.arange(len(labels))[::-1]; ax.barh(y, vals, height=0.62, color=DK)
    for yy, v in zip(y, vals): ax.text(v + max(vals) * 0.012, yy, fmt.format(v), va="center", fontsize=10, fontweight="bold", color=INK)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=10, color=MUT); ax.xaxis.set_visible(False)
    ax.spines["bottom"].set_visible(False); ax.set_xlim(0, max(vals) * 1.15); save(f, name)

def hgroup(name, labels, a, b, fmt="{:+.0f}%", la="New", lb="Win-back", fs=9, lab_fs=9.5, legend_loc="lower right"):
    f, ax = fig(name); y = np.arange(len(labels))[::-1]; bh = 0.36
    a, b = np.asarray(a, float), np.asarray(b, float)
    ax.barh(y + bh / 2, a, height=bh, color=LT, label=la); ax.barh(y - bh / 2, b, height=bh, color=DK, label=lb)
    lo, hi = min(0, np.nanmin(a), np.nanmin(b)), max(np.nanmax(a), np.nanmax(b)); rng = hi - lo
    for yy, v, bold in [(yy, v, False) for yy, v in zip(y + bh / 2, a)] + [(yy, v, True) for yy, v in zip(y - bh / 2, b)]:
        ax.text(v + (rng * 0.012 if v >= 0 else -rng * 0.012), yy, fmt.format(v), va="center", ha="left" if v >= 0 else "right",
                fontsize=fs, color=INK, fontweight="bold" if bold else "normal")
    if lo < 0: ax.axvline(0, color=MID, lw=1)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=lab_fs, color=MUT); ax.xaxis.set_visible(False)
    ax.spines["bottom"].set_visible(False); ax.spines["left"].set_visible(False)
    ax.set_xlim(lo - rng * (0.14 if lo < 0 else 0.0), hi + rng * 0.16)
    ax.legend(frameon=False, fontsize=9.5, labelcolor=MUT, loc=legend_loc); save(f, name)

def pair(df, key, val, order=None):
    pv = df.pivot_table(index=key, columns="start_status", values=val, aggfunc="sum")
    if order is not None: pv = pv.reindex(order)
    return pv.get("new", pd.Series(np.nan, index=pv.index)).fillna(0).values, pv.get("winback", pd.Series(np.nan, index=pv.index)).fillna(0).values, list(pv.index)

def draw_python_charts():
    stacked_months("c1", subs_m.drop(columns="total"), avg=M["hist_avg"])
    if T["rev"] is not None:
        stacked_months("c2", T["rev"].pivot_table(index="m", columns="start_status", values="renewal_value_forgone_usd", fill_value=0).sort_index(), money=True)
    tiers_order = sorted(ti.mou_tier.unique())
    tp = ti.pivot_table(index="start_status", columns="mou_tier", values="subs", aggfunc="sum", fill_value=0)
    stack100("c3", ["New", "Win-back"], {"New": [tp.loc["new"].get(t, 0) for t in tiers_order], "Win-back": [tp.loc["winback"].get(t, 0) for t in tiers_order]},
             [t.split(" ", 1)[1].capitalize() for t in tiers_order], G6[-len(tiers_order):] if len(tiers_order) <= 6 else G6)
    stack100("c4", ["New", "Win-back"], {"New": [att.loc["new"].get(c, 0) for c in (OWN, OTHER, NONE)], "Win-back": [att.loc["winback"].get(c, 0) for c in (OWN, OTHER, NONE)]},
             ["Watched on this subscription", "Watched only on another subscription / pass", "Did not watch"], [DK, MID, LT], fs=11)
    top7 = by_c.head(7); hbar("c5", list(top7.index), list(top7.values))
    if T["nw_problems"] is not None:
        npb = T["nw_problems"].set_index("start_status")
        cols = [("never_opened_player", "Never opened the player"), ("had_error_coded_session", "Had an error-coded session"),
                ("had_entitlement_error", "Had an entitlement error"), ("opened_no_error_no_play", "Opened, no error, no play")]
        a = [npb.loc["new", c] / npb.loc["new", "non_watching_subs"] * 100 for c, _ in cols]
        b = [npb.loc["winback", c] / npb.loc["winback", "non_watching_subs"] * 100 for c, _ in cols]
        hgroup("c5b", [l for _, l in cols], a, b, fmt="{:.0f}%", fs=10, lab_fs=10.5)
    f, ax = fig("c6")
    for st, col in (("winback", DK), ("new", MID)):
        g = ev[ev.start_status == st].sort_values("offset_days")
        ax.plot(g.offset_days, g.avg_cancels_per_peak_day, color=col, lw=2.4, marker="o", ms=5, label=LAB[st])
        v0 = g[g.offset_days == 0].avg_cancels_per_peak_day
        if len(v0): ax.text(0.2, v0.iloc[0] * 1.02, f"{v0.iloc[0]:,.0f}", fontsize=11, fontweight="bold", color=INK)
    ymax = ev.avg_cancels_per_peak_day.max() * 1.15
    ax.axvline(0, color=INK, ls="--", lw=1); ax.text(0.15, ymax * 0.95, "Match day", fontsize=10, color=INK)
    xs = sorted(ev.offset_days.unique()); ax.set_xticks(xs); ax.set_xticklabels([f"{int(i):+d}" if i else "0" for i in xs], fontsize=10)
    ax.set_ylim(0, ymax); ax.yaxis.grid(True, color=GRID, lw=.6); ax.set_axisbelow(True); ax.spines["left"].set_visible(False)
    ax.set_xlabel("Days relative to peak match day", fontsize=10, color=MUT); ax.set_ylabel("Cancellations per match day", fontsize=10, color=MUT)
    ax.legend(frameon=False, fontsize=10, labelcolor=MUT, loc="lower right", ncol=2); save(f, "c6")
    order = ct.drop_duplicates("competitionName").sort_values("viewers", ascending=False)["competitionName"].tolist()
    a, b, labs = pair(ct, "competitionName", "uplift_vs_pre_pct", order); hgroup("c7", labs, a, b)
    mt2 = mt.copy(); mt2["label"] = mt2.apply(match_label, axis=1)
    order = mt2.drop_duplicates("label").sort_values("viewers", ascending=False)["label"].tolist()
    a, b, labs = pair(mt2, "label", "uplift_vs_pre_pct", order); hgroup("c8", labs, a, b, fs=8.5, lab_fs=8.5)
    if T["wc_period"] is not None:
        f, ax = fig("c9"); xx = np.arange(3); bw = 0.36
        a = [wp.loc["new", p] for p in ("1 Before", "2 During", "3 After")]; b = [wp.loc["winback", p] for p in ("1 Before", "2 During", "3 After")]
        ax.bar(xx - bw / 2, a, bw, color=LT, label="New"); ax.bar(xx + bw / 2, b, bw, color=DK, label="Win-back")
        for i in range(3):
            ax.text(xx[i] - bw / 2, a[i] * 1.01, f"{a[i]:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK)
            ax.text(xx[i] + bw / 2, b[i] * 1.01, f"{b[i]:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK)
        ax.set_xticks(xx); ax.set_xticklabels(["4 weeks\nbefore", f"During\n({M['wc_days']} match days)", "4 weeks\nafter"], fontsize=10, color=MUT)
        ax.yaxis.set_visible(False); ax.spines["left"].set_visible(False); ax.set_ylim(0, max(a + b) * 1.2)
        ax.legend(frameon=False, fontsize=10, labelcolor=MUT, loc="upper right"); save(f, "c9")
        w2 = T["wc_match_tbl"].copy()
        w2["label"] = w2.apply(lambda r: f"{r['home_team']} v {r['away_team']}" + (" (final)" if D(r["match_date"]) == M["wc_to"] else ""), axis=1)
        order = w2.drop_duplicates("label").sort_values("viewers", ascending=False)["label"].tolist()
        a, b, labs = pair(w2, "label", "uplift_vs_pre_pct", order); hgroup("c10", labs, a, b, fs=8, lab_fs=8.5)
    oc_order = sorted(oc.outcome.unique())
    rows = {f"{LAB[s]}\n({n0(M['oc_n'][i])} customers)": [oc[(oc.start_status == s) & (oc.outcome == o)]["customers"].sum() for o in oc_order]
            for i, s in enumerate(("new", "winback"))}
    stack100("c11", list(rows), rows, [o.split(" ", 1)[1] for o in oc_order], ["#3F4756", MID, "#8E96A6", LT], fs=9.5, minshare=0.08, counts=True)
    a, b, labs = pair(rc, "next_channel", "returns", rc.groupby("next_channel")["returns"].sum().sort_values(ascending=False).index.tolist())
    hgroup("c12", labs, a, b, fmt="{:,.0f}", fs=9, lab_fs=10)
    top_rc = rcp.groupby("competitionName")["returners"].sum().sort_values(ascending=False).head(7).index.tolist()
    a, b, labs = pair(rcp, "competitionName", "returners", top_rc); hgroup("c13", labs, a, b, fmt="{:,.0f}", fs=8.5, lab_fs=9)
    if T["cancel_speed"] is not None:
        cs = T["cancel_speed"]
        f, ax = fig("c14"); cats = ["within_1d", "within_7d", "within_30d"]; xx = np.arange(3); bw = 0.2
        series = [("History", "new", "#E6E9EF", "New, history"), ("Review window", "new", LT, f"New, {WIN}"),
                  ("History", "winback", MID, "Win-back, history"), ("Review window", "winback", DK, f"Win-back, {WIN}")]
        for j, (per, st, col, lab) in enumerate(series):
            r = cs[(cs.period == per) & (cs.start_status == st)]
            if not len(r): continue
            vals = [r.iloc[0][c] * 100 for c in cats]
            ax.bar(xx + (j - 1.5) * bw, vals, bw * 0.92, color=col, label=lab)
            for x0, v in zip(xx + (j - 1.5) * bw, vals): ax.text(x0, v + 1, f"{v:.0f}%", ha="center", fontsize=8.5, color=INK)
        ax.set_xticks(xx); ax.set_xticklabels(["Within 1 day", "Within 7 days", "Within 30 days"], fontsize=10, color=MUT)
        ax.yaxis.set_visible(False); ax.spines["left"].set_visible(False); ax.set_ylim(0, 115)
        ax.legend(frameon=False, fontsize=9, labelcolor=MUT, ncol=2, loc="upper left"); save(f, "c14")
    if T["winback_cycles"] is not None:
        cy = T["winback_cycles"].copy(); order = ["1", "2", "3", "4-5", "6+"]
        cy["bucket"] = cy["bucket"].astype(str); cy = cy.set_index("bucket").reindex(order).fillna(0)
        f, ax = fig("c15"); xx = np.arange(len(order)); tot = cy["customers"].sum()
        ax.bar(xx, cy["customers"], color=DK, width=0.6)
        for i, v in enumerate(cy["customers"]): ax.text(i, v * 1.01, f"{v:,.0f}\n{v/tot:.0%}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK)
        ax.set_xticks(xx); ax.set_xticklabels(["1*", "2", "3", "4-5", "6+"], fontsize=11, color=MUT); ax.set_xlabel("Lifetime D2C subscriptions per customer", fontsize=10, color=MUT)
        ax.yaxis.set_visible(False); ax.spines["left"].set_visible(False); ax.set_ylim(0, cy["customers"].max() * 1.25); save(f, "c15")
    if T["return_speed"] is not None:
        r = T["return_speed"].set_index("start_status")
        cols = ["d0_7", "d8_14", "d15_30", "later", "not_back"]
        rows = {LAB[s]: [r.loc[s, c] for c in cols] for s in ("new", "winback") if s in r.index}
        stack100("c16", list(rows), rows, ["Back within 7 days", "8-14 days", "15-30 days", "More than 30 days", "Not back (yet)"],
                 ["#3F4756", DK, "#8E96A6", "#B3BAC7", "#E6E9EF"], fs=10)

CHART_SOURCE = P.get("CHART_SOURCE", "python")
draw_python_charts() if CHART_SOURCE == "python" else None
missing = [c for c in SIZE if not os.path.exists(os.path.join(CHART_DIR, c + ".png"))]
if CHART_SOURCE != "python" and missing:
    print("ggplot2 PNGs missing for", missing, "- drawing those with matplotlib instead")
    draw_python_charts()
print("Charts in", CHART_DIR, ":", sorted(f for f in os.listdir(CHART_DIR) if f.endswith(".png")))

In [0]:
%python
def rgb(h): return RGBColor.from_string(h.replace("#", ""))
C_INK, C_GOLD, C_DGOLD, C_MUT, C_BG, C_CARD, C_LINE, C_RED, C_SOFT = "222B3A", "F2B705", "A87B00", "5B6577", "FAFBFD", "FFFFFF", "E3E7EE", "C8453B", "F6E7B0"
FONT = "Calibri"
FOOTER = f"TOD AI & Data Analytics  |  Voluntary cancellations, {WIN}  |  Data as of {ASOF_TXT}"

prs = Presentation()
prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
BLANK = prs.slide_layouts[6]

def rect(s, x, y, w, h, fill, line=None, shape=MSO_SHAPE.RECTANGLE):
    sh = s.shapes.add_shape(shape, Inches(x), Inches(y), Inches(w), Inches(h))
    sh.fill.solid(); sh.fill.fore_color.rgb = rgb(fill)
    if line: sh.line.color.rgb = rgb(line); sh.line.width = Pt(0.75)
    else: sh.line.fill.background()
    st = sh._element.find(qn("p:style"))          # drop the theme style so no shadow or outline is inherited
    if st is not None: sh._element.remove(st)
    return sh

def text(s, x, y, w, h, runs, size=14, color=C_INK, bold=False, italic=False, align=PP_ALIGN.LEFT, anchor=MSO_ANCHOR.TOP, bullets=False, space_after=0):
    """runs: a string, or a list of paragraphs where each paragraph is a string or a list of (text, bold) pieces."""
    tb = s.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)); tf = tb.text_frame
    tf.word_wrap = True; tf.vertical_anchor = anchor
    tf.margin_left = tf.margin_right = tf.margin_top = tf.margin_bottom = 0
    paras = [runs] if isinstance(runs, str) else runs
    for i, para in enumerate(paras):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        if space_after: p.space_after = Pt(space_after)
        pieces = [(para, bold)] if isinstance(para, str) else para
        for t, b in pieces:
            r = p.add_run(); r.text = t; f = r.font
            f.size, f.bold, f.italic, f.name = Pt(size), b, italic, FONT; f.color.rgb = rgb(color)
        if bullets:
            pPr = p._p.get_or_add_pPr(); pPr.set("marL", str(Emu(Inches(0.25)))); pPr.set("indent", str(-Emu(Inches(0.25))))
            bu = pPr.makeelement(qn("a:buChar"), {"char": "•"}); pPr.append(bu)
    return tb

def slide(tab, title):
    s = prs.slides.add_slide(BLANK)
    s.background.fill.solid(); s.background.fill.fore_color.rgb = rgb(C_BG)
    rect(s, 0.5, 0, 2.7, 0.4, C_GOLD)
    text(s, 0.5, 0, 2.7, 0.4, tab, size=11, bold=True, align=PP_ALIGN.CENTER, anchor=MSO_ANCHOR.MIDDLE)
    text(s, 0.5, 0.6, 12.3, 1.05, title, size=26, bold=True)
    text(s, 0.5, 7.05, 10, 0.3, FOOTER, size=10, color=C_MUT)
    text(s, 12.3, 7.05, 0.6, 0.3, str(len(prs.slides)), size=10, color=C_MUT, align=PP_ALIGN.RIGHT)
    return s

def card(s, x, y, w, h, fill=C_CARD): return rect(s, x, y, w, h, fill, C_LINE)

def stat(s, x, y, w, big, label, color=C_INK, h=1.5):
    card(s, x, y, w, h)
    text(s, x + 0.2, y + 0.1, w - 0.4, h * 0.5, big, size=30, bold=True, color=color, anchor=MSO_ANCHOR.MIDDLE)
    text(s, x + 0.2, y + h * 0.58, w - 0.4, h * 0.4, label, size=13, color=C_MUT)

def callout(s, x, y, w, h, t):
    card(s, x, y, w, h, C_SOFT); text(s, x + 0.2, y + 0.05, w - 0.4, h - 0.1, t, size=14, bold=True, anchor=MSO_ANCHOR.MIDDLE)

def note(s, t, y=6.6): text(s, 0.5, y, 12.3, 0.4, t, size=11, italic=True, color=C_MUT)
def sub(s, t, x, y, w): text(s, x, y, w, 0.4, t, size=16, bold=True)

def pic(s, name, x, y, alt):
    w, h = SIZE[name]
    p = s.shapes.add_picture(os.path.join(CHART_DIR, name + ".png"), Inches(x), Inches(y), Inches(w), Inches(h))
    p._element.nvPicPr.cNvPr.set("descr", alt)
    return p

def numbered_cards(s, items, y0=1.9):
    for i, (head, body) in enumerate(items):
        x, y = 0.5 + (i % 2) * 6.25, y0 + (i // 2) * 2.45
        card(s, x, y, 6.05, 2.25)
        rect(s, x + 0.25, y + 0.25, 0.6, 0.6, C_GOLD, shape=MSO_SHAPE.OVAL)
        text(s, x + 0.25, y + 0.25, 0.6, 0.6, str(i + 1), size=20, bold=True, align=PP_ALIGN.CENTER, anchor=MSO_ANCHOR.MIDDLE)
        text(s, x + 1.05, y + 0.25, 4.8, 0.6, head, size=18, bold=True, anchor=MSO_ANCHOR.MIDDLE)
        text(s, x + 0.25, y + 1.0, 5.55, 1.15, body, size=14, color=C_MUT)

In [0]:
%python
def safe(fn):
    try: fn()
    except Exception as e: print(f"Skipped slide {fn.__name__}: {type(e).__name__}: {e}")

def s_cover():
    s = prs.slides.add_slide(BLANK); s.background.fill.solid(); s.background.fill.fore_color.rgb = rgb(C_INK)
    rect(s, 0.8, 0, 3.2, 0.55, C_GOLD)
    text(s, 0.8, 0, 3.2, 0.55, "Customer Retention Deep Dive", size=12, bold=True, align=PP_ALIGN.CENTER, anchor=MSO_ANCHOR.MIDDLE)
    text(s, 0.8, 1.9, 11.2, 1.9, f"Voluntary cancellations, {WIN}: who cancelled, why, and what happened next", size=40, bold=True, color="FFFFFF")
    text(s, 0.8, 4.0, 10, 0.8, "What changed, whether it is normal behaviour, and what customers did after cancelling", size=20, color="D9DEE8")
    text(s, 0.8, 6.5, 11, 0.4, f"D2C subscriptions  |  Data as of {ASOF_TXT}  |  TOD AI & Data Analytics", size=14, color=C_GOLD)

def s_exec():
    below = M["last_vs_avg"] < 0
    s = slide("Executive summary", f"{WIN} cancellations ran {'below' if below else 'above'} the usual run rate; "
              f"the bigger issues are non-watchers and end-of-competition exits")
    stat(s, 0.5, 1.85, 2.9, n0(M["subs"]), f"subscriptions cancelled in {WIN} ({n0(M['cust'])} customers)", h=1.65)
    stat(s, 3.6, 1.85, 2.9, pc(M["wb_share"]), f"started as Win-back ({n0(M['subs_wb'])}); {pc(1-M['wb_share'])} started as New ({n0(M['subs_new'])})", h=1.65)
    stat(s, 6.7, 1.85, 2.9, sp(100 * M["last_vs_avg"]), f"{mlabel(M['last_m'])} volume ({k(M['last_v'])}) vs the {k(M['hist_avg'])} monthly average", C_DGOLD, h=1.65)
    stat(s, 9.8, 1.85, 3.0, pc(M["never_share"]), f"never watched ({n0(M['never'])}); {pc(M['never_wb_sh'])} of win-backs vs {pc(M['never_new_sh'])} of new", C_RED, h=1.65)
    vol = ", ".join(f"{mon(D(m+'-01'))} {k(subs_m.loc[m,'total'])}" for m in win_months)
    rev_txt = f" Renewal value forgone was {usd(M['rev_win'])}." if "rev_win" in M else ""
    wc_txt = f"; they fell {pc(-M['wc_chg_new'])} (New) and {pc(-M['wc_chg_wb'])} (Win-back) during the World Cup" if "wc_chg_new" in M else ""
    pts = [[("Volume: ", True), (f"{vol}, against a {k(M['hist_avg'])} average; the peak was {mlabel(M['peak_m'])} at {k(M['peak_v'])}.{rev_txt}", False)],
           [("Behaviour: ", True), (f"{pc(M['other_wb'])} of cancelling win-backs watched only on another subscription or pass, and {pc(M['never_wb_sh'])} of win-backs never watched at all.", False)],
           [("Football: ", True), (f"cancellations rise on peak match days ({sp(100*M['ev_up_new'])} New) and most on big finals ({M['match_top']}: {sp(M['match_top_new'][2])} New){wc_txt}.", False)],
           [("After: ", True), (f"{pc(M['oc_stop'][0])} of New and {pc(M['oc_stop'][1])} of Win-back customers stop watching before expiry; {pc(M['ret_top_sh'])} of win-back returns come through {M['ret_top_ch']}.", False)]]
    card(s, 0.5, 3.75, 12.3, 3.1)
    text(s, 0.8, 3.9, 11.7, 2.8, pts, size=16, bullets=True, space_after=8)

def s_approach():
    s = slide("Approach", f"Every subscription is classified by the status it started with, so {WIN} and history compare like for like")
    items = [("Cancelled D2C subscriptions", f"Paid D2C subscriptions with a cancellation date from {FROM.day} {mon(FROM)} to {TO.day} {mon(TO)} {TO.year}. Excludes promo, non-commercial, daily/weekly and product passes."),
             ("Start status", "Each subscription starts as New or Win-back and ends as Churned, so the status at the first version says what kind of customer it was."),
             ("Viewing by subscription", "Viewing is matched to the subscription that was cancelled, not just the customer, so passes and other subscriptions are not mistaken for use."),
             ("Match days and outcomes", f"Cancellations are compared with match days and tracked through to paid expiry and any return, as of {ASOF_TXT}.")]
    for i, (h, b) in enumerate(items):
        x = 0.5 + i * 3.1; card(s, x, 1.95, 2.9, 3.7)
        rect(s, x + 0.25, 2.2, 0.6, 0.6, C_GOLD, shape=MSO_SHAPE.OVAL)
        text(s, x + 0.25, 2.2, 0.6, 0.6, str(i + 1), size=20, bold=True, align=PP_ALIGN.CENTER, anchor=MSO_ANCHOR.MIDDLE)
        text(s, x + 0.25, 2.95, 2.45, 0.7, h, size=18, bold=True)
        text(s, x + 0.25, 3.7, 2.45, 1.9, b, size=14, color=C_MUT)
    callout(s, 0.5, 5.9, 12.3, 0.85, "Football-driven cycling and product or commercial problems are hypotheses in this deck, not proven causes.")

def s_monthly():
    rel = "below" if M["last_vs_avg"] < 0 else "above"
    vol = " and ".join(f"{k(subs_m.loc[m,'total'])} in {mon(D(m+'-01'))}" for m in win_months[-2:])
    s = slide("What changed", f"Cancellations in {WIN} ran {rel} the {M['hist_label']} average: {vol} vs {k(M['hist_avg'])}")
    pic(s, "c1", 0.5, 1.85, "Monthly cancelling subscriptions by start status")
    lo, hi = M["wb_share_win"]; hlo, hhi = M["wb_share_hist"]
    stat(s, 9.5, 1.85, 3.3, n0(M["subs"]), f"subscriptions cancelled in {WIN}", h=1.45)
    stat(s, 9.5, 3.45, 3.3, f"{pc(lo)}-{pc(hi)}" if round(lo, 2) != round(hi, 2) else pc(lo),
         f"Win-back share in {WIN} ({pc(hlo)}-{pc(hhi)} in history)", C_DGOLD, h=1.45)
    stat(s, 9.5, 5.05, 3.3, k(M["peak_v"]), f"{mlabel(M['peak_m'])}, the highest month in the period", h=1.45)
    z = (f" Customer-level check: {mlabel(M['last_m'])} = {n0(M['cust_last'])} vs a monthly mean of {n0(M['cust_mean'])} (z = {M['z']:.2f})." if "z" in M else "")
    note(s, f"Cancelling subscriptions by month of cancellation; {WIN} in full colour.{z}")

def s_revenue():
    s = slide("Revenue", f"Renewal value forgone in {WIN} was {usd(M['rev_win'])}; the average last charge was "
              f"${M['avg_charge_win']:.0f} vs ${M['avg_charge_hist']:.0f} in earlier months")
    pic(s, "c2", 0.5, 1.85, "Renewal value forgone by month")
    stat(s, 9.5, 1.85, 3.3, usd(M["rev_win"]), f"renewal value that will not be billed, {WIN}", h=1.45)
    stat(s, 9.5, 3.45, 3.3, f"${M['avg_charge_win']:.0f} vs ${M['avg_charge_hist']:.0f}", f"average last charge, {WIN} vs earlier months", C_DGOLD, h=1.45)
    stat(s, 9.5, 5.05, 3.3, usd(M["rev_peak_v"]), f"{mlabel(M['rev_peak_m'])}, the highest month", C_MUT, h=1.45)
    note(s, "Value = last charge (USD, incl. tax) on each cancelled subscription. An estimate of the renewal not billed, not a realised loss.")

def s_never():
    ratio = M["never_wb_sh"] / M["never_new_sh"] if M["never_new_sh"] else np.nan
    rel = "twice as likely" if 1.8 <= ratio <= 2.2 else f"{ratio:.1f}x as likely"
    s = slide("Non-watching", f"{pc(M['never_share'])} of cancelling subscriptions never watched, and win-backs are {rel} as New to be non-watchers "
              f"({pc(M['never_wb_sh'])} vs {pc(M['never_new_sh'])})")
    sub(s, "Minutes watched in the subscription window, any subscription or pass", 0.5, 1.8, 8.3)
    pic(s, "c3", 0.5, 2.25, "Viewing depth tiers")
    stat(s, 9.2, 1.85, 3.6, n0(M["never"]), f"subscriptions never watched ({pc(M['never_share'],1)} of {n0(M['subs'])})", C_RED)
    stat(s, 9.2, 3.55, 3.6, f"{pc(M['never_wb_sh'])} vs {pc(M['never_new_sh'])}", f"Win-back ({n0(M['never_wb'])}) vs New ({n0(M['never_new'])}) share who never watched")
    callout(s, 0.5, 5.8, 8.3, 0.85, f"At the other end, {pc(M['heavy_sh'].get('new',0))} of New and {pc(M['heavy_sh'].get('winback',0))} of Win-back watched 20+ hours before cancelling.")
    note(s, "Never watched = no viewing minute in the window on this or any other subscription or pass.", 6.75)

def s_attr():
    s = slide("Viewing attribution", f"{pc(M['other_wb'])} of cancelling win-backs watched only on a different subscription or pass, so the one they cancelled went unused")
    sub(s, "Where cancelling subscriptions' viewing took place", 0.5, 1.8, 8.3)
    pic(s, "c4", 0.5, 2.25, "Viewing attribution")
    stat(s, 9.2, 1.85, 3.6, pc(M["other_wb"], 1), f"of win-backs ({n0(M['other_wb_n'])}) watched only elsewhere vs {pc(M['other_new'],1)} of New ({n0(M['other_new_n'])})", C_RED)
    stat(s, 9.2, 3.55, 3.6, f"{pc(M['own_wb'])} vs {pc(M['own_new'])}", "watched on the cancelled subscription: Win-back vs New")
    callout(s, 0.5, 5.8, 8.3, 0.85, "Win-backs often move to another pass or live subscription, so a customer-level view overstates their use.")
    note(s, "Viewing is matched on subscription ID. Passes are the likely 'other' product; this has not been verified product by product.", 6.75)

def s_nw_country():
    prods = ", ".join(M["top_products"][:-1]) + " and " + M["top_products"][-1] if len(M["top_products"]) > 1 else M["top_products"][0]
    s = slide("Non-watchers", f"Non-watching cancellers cluster in {M['top3_c'][0]} and {M['top3_c'][1]}, mostly on the {prods} plans")
    pic(s, "c5", 0.5, 1.85, "Non-watching subscriptions by country")
    stat(s, 9.2, 1.85, 3.6, n0(M["top_c_v"]), f"{M['top_c']} non-watching subscriptions, {pc(M['top_c_v']/M['nw_total'])} of {n0(M['nw_total'])}")
    stat(s, 9.2, 3.55, 3.6, pc(M["top3_sh"]), f"come from {', '.join(M['top3_c'][:2])} and {M['top3_c'][2]} together", C_DGOLD)
    cells_txt = "; ".join(f"{p} in {c} ({n0(v)})" for p, c, v in M["top_cells"])
    callout(s, 9.2, 5.25, 3.6, 1.3, f"Largest cells: {cells_txt}.")
    note(s, "Top 7 countries by non-watching subscriptions in the cohort.")

def s_nw_problems():
    npb = T["nw_problems"].set_index("start_status")
    sh = lambda st, c: npb.loc[st, c] / npb.loc[st, "non_watching_subs"]
    s = slide("Non-watchers", f"Most non-watchers never opened the player ({pc(sh('new','never_opened_player'))} of New, "
              f"{pc(sh('winback','never_opened_player'))} of Win-back); errors explain only part of it")
    sub(s, "Share of non-watching subscriptions with each signal (signals can overlap)", 0.5, 1.8, 8.3)
    pic(s, "c5b", 0.5, 2.2, "Non-watcher signals")
    stat(s, 9.2, 1.85, 3.6, pc(sh("winback", "had_error_coded_session")), "of Win-back non-watchers had an error-coded session", C_RED)
    stat(s, 9.2, 3.55, 3.6, pc(sh("winback", "had_entitlement_error")), "had an entitlement error", C_DGOLD)
    callout(s, 9.2, 5.25, 3.6, 1.3, "Never opening the player points to intent or onboarding, not playback failure.")
    note(s, "Sessions inside each subscription's window, any subscription. A subscription can show more than one signal.", 6.75)

def s_speed():
    cs = T["cancel_speed"]; g = lambda per, st, c: cs[(cs.period == per) & (cs.start_status == st)].iloc[0][c]
    w1 = g("Review window", "new", "within_1d"); h1 = g("History", "new", "within_1d")
    s = slide("Cancel speed", f"{pc(w1)} of New subscriptions in {WIN} were cancelled within a day of sign-up, vs {pc(h1)} in history")
    pic(s, "c14", 0.5, 1.85, "Speed of cancellation after sign-up")
    stat(s, 9.2, 1.85, 3.6, f"{g('Review window','winback','median_days'):.0f} days", f"median days from start to cancel, Win-back, {WIN}")
    stat(s, 9.2, 3.55, 3.6, f"{g('Review window','new','median_days'):.0f} days", f"median days from start to cancel, New, {WIN}", C_DGOLD)
    callout(s, 9.2, 5.25, 3.6, 1.3, "Fast cancels usually mean switching off auto-renewal, not leaving at once.")
    note(s, "Share of cancelled subscriptions cancelled within 1, 7 and 30 days of their start date.", 6.65)

def s_event():
    s = slide("Football timing", f"Cancellations rise on peak match days ({sp(100*M['ev_up_new'])} New, {sp(100*M['ev_up_wb'])} Win-back) "
              f"and fall back the next day")
    pic(s, "c6", 0.5, 1.85, "Cancellations around peak match days")
    stat(s, 9.2, 1.85, 3.6, f"{M['d0_new']:,.0f} vs {M['avg_new']:,.0f}", "New: match-day cancellations vs the average day")
    stat(s, 9.2, 3.55, 3.6, f"{M['d0_wb']:,.0f} vs {M['avg_wb']:,.0f}", "Win-back: match-day cancellations vs the average day", C_DGOLD)
    callout(s, 9.2, 5.25, 3.6, 1.3, "A one-day effect: no wave of cancellations in the week after." if M["post_new_vs_avg"] < 0.05 else "Cancellations stay raised in the week after the match.")
    note(s, f"Peak match day = busiest 25% of match days by viewers ({M['peak_days']} of {M['match_days']}, {mon(EV_FROM)} {EV_FROM.year} onwards). Overlapping windows count some cancellations twice. Timing, not proof of cause.")

def s_comp():
    s = slide("By competition", f"{M['comp_top']} match days lift New cancellations by {M['comp_top_up']:.0f}%; "
              f"{' and '.join(M['busiest'])} barely move them ({M['bus_lo']:+.0f}% to {M['bus_hi']:+.0f}%)")
    sub(s, "Match-day uplift vs the average of the 3 days before", 0.5, 1.8, 8.3)
    pic(s, "c7", 0.5, 2.2, "Uplift by competition")
    stat(s, 9.2, 1.85, 3.6, sp(M["comp_top_up"]), f"{M['comp_top']}, New customers ({M['comp_top_days']} match days)")
    stat(s, 9.2, 3.55, 3.6, sp(M["comp_wb_top_up"]), f"{M['comp_wb_top']}, Win-back customers (highest Win-back uplift)", C_DGOLD)
    callout(s, 9.2, 5.25, 3.6, 1.3, "Competitions with match days almost every day show no clear peak.")
    note(s, "Top 10 competitions by viewers, excluding the FIFA World Cup.", 6.65)

def s_match():
    pn, dn, un = M["match_top_new"]; pw, dw, uw = M["match_top_wb"]
    s = slide("By match", f"{M['match_top']} stands out: match-day cancellations {'jumped' if un > 0 else 'changed'} {abs(un):.0f}% for New and "
              f"{'rose' if uw >= 0 else 'fell'} {abs(uw):.0f}% for Win-back")
    sub(s, "Match-day uplift vs the average of the 3 days before", 0.5, 1.8, 8.3)
    pic(s, "c8", 0.5, 2.2, "Uplift for the 10 most-watched matches")
    stat(s, 9.2, 1.85, 3.6, f"{pn:,.0f} to {dn:,.0f}", "New cancellations per day: 3-day average before vs match day")
    stat(s, 9.2, 3.55, 3.6, f"{pw:,.0f} to {dw:,.0f}", "Same match: Win-back cancellations per day", C_DGOLD)
    callout(s, 9.2, 5.25, 3.6, 1.3, f"{M['match_main_n']} of the {M['match_n']} most-watched matches (excl. World Cup) are {M['match_main_comp']} games.")
    note(s, "Day of week was not checked, so part of a spike may be a weekend pattern. Excludes World Cup matches.", 6.65)

def s_wc():
    fin = M.get("wc_final")
    tail = f"; only the final spiked (New {sp(fin[1])})" if fin and fin[1] > 25 else ""
    s = slide("FIFA World Cup", f"During the World Cup cancellations fell {pc(-M['wc_chg_new'])} (New) and {pc(-M['wc_chg_wb'])} (Win-back){tail}")
    sub(s, "Average cancellations per day", 0.5, 1.8, 6.0); pic(s, "c9", 0.5, 2.2, "Daily cancellations before, during and after the World Cup")
    sub(s, "Match-day uplift, top 10 World Cup matches", 6.6, 1.8, 6.2); pic(s, "c10", 6.5, 2.2, "Uplift for top 10 World Cup matches")
    note(s, f"Change during the tournament vs the 4 weeks before. {M['wc_days']} match days, {M['wc_from'].day} {mon(M['wc_from'])}-{M['wc_to'].day} {mon(M['wc_to'])} {M['wc_to'].year}.", 6.65)

def s_cycles():
    cy = T["winback_cycles"].copy(); cy["bucket"] = cy["bucket"].astype(str); tot = cy["customers"].sum()
    multi = cy[~cy.bucket.isin(["1", "2"])]["customers"].sum() / tot
    s = slide("Repeat cycling", f"{pc(multi)} of cancelling win-backs have three or more lifetime subscriptions behind them")
    pic(s, "c15", 0.5, 1.85, "Lifetime subscriptions of cancelling win-backs")
    stat(s, 9.2, 1.85, 3.6, n0(tot), "cancelling win-back customers")
    stat(s, 9.2, 3.55, 3.6, pc(cy[cy.bucket == "6+"]["customers"].sum() / tot), "have six or more lifetime subscriptions", C_DGOLD)
    callout(s, 9.2, 5.25, 3.6, 1.3, "Repeat subscribe-cancel-return cycles are normal for this group.")
    note(s, "* Only one subscription in this extract: the earlier one is outside the D2C filter or the data window.", 6.65)

def s_outcomes():
    s = slide("After cancelling", f"{pc(min(M['oc_stop']))}-{pc(max(M['oc_stop']))} of cancelling customers stop watching before expiry; "
              f"only {pc(min(M['oc_newsub']))}-{pc(max(M['oc_newsub']))} start a new subscription early")
    pic(s, "c11", 0.5, 1.85, "Outcomes between cancelling and paid expiry")
    stat(s, 9.2, 1.85, 3.6, f"{pc(M['oc_newsub'][0],1)} / {pc(M['oc_newsub'][1],1)}", "New / Win-back who started a new subscription before expiry")
    stat(s, 9.2, 3.55, 3.6, f"{pc(M['oc_kept'][0],1)} / {pc(M['oc_kept'][1],1)}", "New / Win-back who kept watching, then expired", C_DGOLD)
    callout(s, 0.5, 5.7, 8.3, 0.9, f"{n0(M['reversed'])} cancellations were reversed and {n0(M['charged_after'])} of {n0(M['subs'])} subscriptions were charged again after cancelling.")
    note(s, f"One outcome per customer (shares sum to 100%). {n0(M['oc_n'][0])} New and {n0(M['oc_n'][1])} Win-back customers.", 6.7)

def s_returns():
    comps = ", ".join(M["ret_comps"][:-1]) + " and " + M["ret_comps"][-1]
    s = slide("Returns", f"Returns come mostly through {M['ret_top_ch']} ({pc(M['ret_top_sh'])} of win-back returns) and centre on {comps}")
    sub(s, "Returns by channel", 0.5, 1.8, 5.9); pic(s, "c12", 0.5, 2.2, "Returns by channel")
    sub(s, "Returners by competition watched in their first 7 days back", 6.6, 1.8, 6.2); pic(s, "c13", 6.5, 2.2, "Returners by competition")
    note(s, f"A return = a new subscription starting on or after the previous one expired. {n0(M['ret_wb'])} Win-back and {n0(M['ret_new'])} New returns observed as of {ASOF_TXT}.", 6.65)

def s_return_speed():
    r = T["return_speed"].set_index("start_status")
    wk = {s: r.loc[s, "d0_7"] / r.loc[s, "matured_subs"] for s in r.index}
    nb = {s: r.loc[s, "not_back"] / r.loc[s, "matured_subs"] for s in r.index}
    s = slide("Return speed", f"{pc(wk.get('winback',0))} of win-backs are back within a week of expiry; {pc(nb.get('new',0))} of New have not come back")
    sub(s, f"Subscriptions whose expiry is at least {P['MATURE_DAYS']} days before {ASOF_TXT}", 0.5, 1.8, 8.3)
    pic(s, "c16", 0.5, 2.25, "Return speed after expiry")
    stat(s, 9.2, 1.85, 3.6, n0(r["matured_subs"].sum()), "subscriptions with a complete return window")
    stat(s, 9.2, 3.55, 3.6, pc(wk.get("new", 0)), "of New back within 7 days", C_DGOLD)
    note(s, "Only subscriptions whose expiry is old enough to read a return are included.", 6.75)

def s_next():
    s = slide("Next steps", "Retention effort is best spent before competitions end and on getting win-backs to a first watch on the right subscription")
    numbered_cards(s, [("Act before the final", "Target customers acquired for a competition before its final with an offer for the next competition or season."),
                       ("Fix the win-back first watch", f"{pc(M['never_wb_sh'])} of win-backs never watch and {pc(M['other_wb'])} use another pass. Check the hand-off from pass to subscription and the onboarding."),
                       (f"Review {M['top3_c'][0]} and {M['top3_c'][1]}", f"Non-watchers concentrate on the {', '.join(M['top_products'])} plans there. Validate login, entitlement and playback journeys."),
                       (f"Use the {M['ret_top_ch']} route", f"{pc(M['ret_top_sh'])} of win-back returns arrive through {M['ret_top_ch']}. Build return offers there around {M['ret_comps'][0]}.")])
    note(s, "These are hypotheses to test with Product and Commercial; the data shows when and who, not why.", 6.7)

def s_appendix():
    s = slide("Appendix", "Definitions and data notes")
    rr = f"{M['hist_label']} mean of {n0(M['cust_mean'])} customers a month (sd {n0(M['cust_sd'])}); {mlabel(M['last_m'])} = {n0(M['cust_last'])} (z = {M['z']:.2f})." if "z" in M else ""
    notes = [[("Cohort: ", True), (f"D2C Subscription-offer subscriptions, latest version, cancellation date {FROM:%d %b %Y} to {TO:%d %b %Y}; excludes non-commercial, daily/weekly and product passes.", False)],
             [("Counts: ", True), (f"{n0(M['subs'])} subscriptions = {n0(M['cust'])} customers, because some customers cancelled more than one subscription.", False)],
             [("Start status: ", True), ("status on the first version of each subscription (New or Win-back), cross-checked against the customer's order of subscriptions.", False)],
             [("Run rate: ", True), (rr + " Monthly chart uses subscriptions.", False)],
             [("Viewing: ", True), ("playing time (ms / 60,000) matched to the subscription ID; 'never watched' = under 1 minute on any subscription in the window.", False)],
             [("Match days: ", True), ("windows of -3 to +7 days overlap and can count a cancellation twice; day of week not checked; timing is not proof of cause.", False)],
             [("Revenue: ", True), ("last charge on each cancelled subscription (USD incl. tax); an estimate of renewal not billed.", False)],
             [("Returns: ", True), ("new subscription on or after the prior expiry; channel and first-7-day views include all observed returns.", False)]]
    card(s, 0.5, 1.8, 12.3, 5.0); text(s, 0.8, 1.95, 11.7, 4.7, notes, size=14, bullets=True, space_after=6)

ORDER = [s_cover, s_exec, s_approach, s_monthly, s_revenue, s_speed, s_never, s_attr, s_nw_country, s_nw_problems,
         s_event, s_comp, s_match, s_wc, s_cycles, s_outcomes, s_returns, s_return_speed, s_next, s_appendix]
NEEDS = {s_revenue: "rev", s_speed: "cancel_speed", s_nw_problems: "nw_problems", s_wc: "wc_period", s_cycles: "winback_cycles", s_return_speed: "return_speed"}
for fn in ORDER:
    if fn in NEEDS and T[NEEDS[fn]] is None:
        print(f"Skipped {fn.__name__}: no {NEEDS[fn]} table"); continue
    safe(fn)
print("Slides built:", len(prs.slides))

In [0]:
%python
fname = f"Voluntary_Cancellations_{FROM:%Y%m%d}_{TO:%Y%m%d}.pptx"
out_path = os.path.join(OUT_ROOT, fname)
prs.save(out_path)
print("Saved:", out_path)
if out_path.startswith("/dbfs/FileStore/"):
    link = "/files/" + out_path[len("/dbfs/FileStore/"):]
    try: displayHTML(f'<a href="{link}" download>Download {fname}</a>')
    except NameError: print("Download link:", link)
else:
    print("Download it from Catalog Explorer (Volumes) or with: databricks fs cp", out_path, ".")